[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exploring-yash/seamless_interaction_dynamics/blob/main/Notebooks/early_warning_signals.ipynb)

# Early Warning Signals for Interaction Collapse

Companion notebook to `measuring_synchrony.ipynb`. Tests whether Critical Slowing Down (CSD) indicators rise before behavioural rupture events on the multimodal coupling signal `CCA_1(t)`.

> **A data scientist's guide to detecting when a conversation is about to break down — using rising variance, autocorrelation, and skewness on a multimodal coupling signal as early warning signs.**

---

## What This Notebook Does

Takes the multimodal coupling signal `CCA_1(t)` from `measuring_synchrony.ipynb` and asks: **does it show statistical warning signs before the interaction ruptures?**

The technique is **Critical Slowing Down (CSD)** — a phenomenon from physics, ecology, and clinical psychology where a system's recovery from small perturbations slows down before it tips into a new regime. Operationally: rising rolling variance, lag-1 autocorrelation, and skewness in the pre-rupture window.

This has direct relevance to:
- **Early warning systems for emerging risks** (Bengio et al. 2024, *Science*)
- **Pre-deployment monitors for human–AI coupling** (Anthropic Responsible Scaling Policy v2.0)
- **Clinical applications** — depression-relapse prediction from momentary affect (Wichers et al. 2016, *JAMA Psychiatry*)

The headline empirical findings: a calibrated H1 null on temporal-rate detection (an honest negative result — see verdict gate below) **alongside** a per-modality classification result that survives a duration-confound audit at AUC = 0.778.

## H1 (pre-registered)

CSD indicators (variance, AR(1), skewness) computed on `CCA_1(t)` rise before independently-defined rupture events (head + gaze composite, disjoint from `CCA_1` inputs), with observed Kendall τ exceeding the 95th percentile of a **circular block bootstrap null** (Kunsch 1989 family; cf. Politis & Romano 1994) in **≥ 2 of 3 primary indicators** for **≥ 70 %** of events, conditional on `N_eff ≥ 100 dyads AND N_events_effective ≥ 200`.

Spectral reddening and ADF stationarity are reported as diagnostic sidecars only. Primary horizon = 60 s (Wichers et al. 2016); 90 s reported as a robustness sidecar. The 30 s horizon was excluded because `WARNING_LEAD_S = 30 s` makes the pre-window zero-length. The legacy 1-of-2 rate is preserved alongside the primary 2-of-3 rule for continuity with earlier methodology.

**Verdict gate.**

| Condition | Label |
|---|---|
| Sample floor not met | `[UNDERPOWERED]` |
| `rate_2of3 ≥ 0.70`, null passed | `[EMPIRICAL]` |
| `0.50 ≤ rate_2of3 < 0.70` | `[EMPIRICAL but weak]` |
| `rate_2of3 < 0.50` | `[NO EFFECT vs NULL]` |
| ≥ 70 % real but partner-shuffle not surpassed | `[COUPLING NOT SPECIFIC]` |

## Contribution framing (methodological + AI-safety)

This notebook targets a **balanced dual contribution**:

1. **Methodological negative result** (primary, BRM-target). Reports a field-normative null for behavioral-only IPS classification at N ≈ 1,700 V00 dyads on the Meta Seamless Interaction dataset, with per-modality decomposition (§9) and non-linear diagnostic (§7) as the methodological deltas. Frames the outcome within Gordon & Bartsch 2026 *Nature Reviews Psychology* IPS-heterogeneity review — our result is not anomalous; field-wide effect sizes on behavioral-only synchrony are consistent with r ≈ 0.12–0.25.

2. **AI-safety proof-of-concept** (secondary, Anthropic-RSP-anchored). Treats the CSD-on-coupling pipeline as a capability-detection prototype: an auditable, calibrated-FPR rupture detector (§8) + a directional-control probe via TE asymmetry (Cell 2H) + an MI-vs-CCA diagnostic that surfaces linearity ceilings (§7). Measurement must precede product (Amodei et al. 2016) — pre-deployment EWS monitors (Bengio et al. 2024) require that we first know what human-human coupling looks like under null conditions.

## AI-safety mapping (Anthropic directions)

Each mapping below is tagged with its evidentiary status:
`[THEORETICAL]` = argued from methodology but not yet tested on this dataset;
(future extension) = target for extension, requires work not in this notebook.

| Direction | NB2 component | Status |
|---|---|---|
| **#2 Measuring alignment** | CSD as brittleness diagnostic on the coupling signal. | `[THEORETICAL]` |
| **#4 AI control** | Transfer-entropy asymmetry on `CCA_1(a,b)` as a directional-influence probe (sidecar). | `[THEORETICAL]` (TE ran but null on this dataset — see §6) |
| **#5 Scalable oversight** | Cross-modal rupture detection with calibrated false-positive rate. | (future extension) (calibration pending 20-dyad Matarić pilot) |
| **#8 Multi-agent** | HMM regime occupancy as a coordination-state tracker (sidecar). | `[THEORETICAL]` (ran, null on this dataset — see §6) |

Primary safety framings: Amodei et al. 2016 (anomaly detection), Bengio et al. 2024 (pre-deployment EWS), Anthropic RSP 2024 (capability-threshold gates).


### Methods-lineage pointer

Inputs come from `measuring_synchrony.ipynb` (CCA_1(t) on body + FAU + head, sklearn). This notebook builds on that multimodal coupling signal — an earlier single-channel Kuramoto baseline on `emotion_valence` alone reached AUC = 0.61, motivating the move to the multimodal coupling architecture.

Alternative framings not implemented here (see §6 limitations + future work): regime-switching via TDE-HMM (Vidaurre et al. 2018) is reported as a sidecar in §2H.2; k-means on wavelet coherence (Kawaguchi et al. 2026) is tracked as a roadmap item.


## 1. Setup & Load Results from companion notebook


In [ ]:
# ----------------------------------------------------------------------------
# Colab: pip install FIRST, then Drive mount AFTER.
# Why: !pip install can invalidate an earlier Drive mount (fuse state
# disruption when newer pkgs are installed that Colab's runtime uses).
# By installing first, the mount is the LAST setup action and stays clean
# through Cell 4 onward.  Run-readiness hardening
# ----------------------------------------------------------------------------

# ============================================================================
# 1A. Install Dependencies  (runs FIRST)
# ============================================================================
!pip install -q scipy numpy matplotlib seaborn pandas scikit-learn statsmodels dtaidistance 'ewstools>=2.1,<2.2' 'hmmlearn>=0.3.0' librosa 'shap>=0.44.0'

# ----------------------------------------------------------------------------
# Drive mount — runs AFTER pip to survive any fuse disruption
# ----------------------------------------------------------------------------
_IN_COLAB = False
try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
    from google.colab import drive as _colab_drive
    try:
        _colab_drive.mount('/content/drive', force_remount=False)
        print('✓ Google Drive mounted')
        # Post-mount sanity: if MyDrive/ isn\'t yet populated, force a clean remount.
        from pathlib import Path as _Path
        if not _Path('/content/drive/MyDrive').exists():
            print('⚠ /content/drive/MyDrive missing after mount — forcing clean remount')
            _colab_drive.mount('/content/drive', force_remount=True)
    except Exception as _e:
        print(f'⚠ Drive mount failed: {_e}')
except ImportError:
    print('ℹ Local environment detected; skipping Drive mount')

# Reproducibility seed
import os as _os
SEED = int(_os.environ.get('SEAMLESS_SEED', 42))
import random as _random
import numpy as _np_seed
_random.seed(SEED)
_np_seed.random.seed(SEED)
try:
    import torch as _torch_seed
    _torch_seed.manual_seed(SEED)
except ImportError:
    pass


In [ ]:
# Run-readiness: parameterized paths
import os
import sys
from pathlib import Path

# Data root: override with SEAMLESS_DATA_DIR, falls back to known locations.
# : auto-retry Drive mount once if Colab drive is unexpectedly absent
# (fixes the restart-trap where !pip install prompts a kernel restart,
# Drive unmounts, and Cell 4 sees no Drive path even though Cell 3 "mounted").
def _resolve_data_root():
    env = os.environ.get('SEAMLESS_DATA_DIR')
    if env and Path(env).exists():
        return Path(env)
    colab_path = Path('/content/drive/MyDrive/seamless_interaction')
    candidates = [
        colab_path,
        Path.home() / 'Documents' / 'seamless_interaction',
        Path.home() / 'Desktop' / 'seamless_interaction',
    ]
    for p in candidates:
        if p.exists():
            return p
    # Colab recovery: if /content/drive exists but the target subfolder is
    # missing, attempt one remount (common when kernel restarted after pip
    # install without auto-remounting). Silently no-op outside Colab.
    try:
        from google.colab import drive as _drv  # type: ignore
        if Path('/content/drive').exists():
            _drv.mount('/content/drive', force_remount=True)
            if colab_path.exists():
                print(f'[recover] remounted Drive; DATA_ROOT = {colab_path}')
                return colab_path
    except Exception:
        pass
    return Path.home() / 'seamless_interaction'  # last-resort default

DATA_ROOT = _resolve_data_root()
if not DATA_ROOT.exists():
    import warnings as _w
    _w.warn(f'DATA_ROOT does not exist: {DATA_ROOT} -- data-loading cells will be empty', RuntimeWarning)
    # Diagnostic listing to help the user figure out what went wrong
    _mydrive = Path('/content/drive/MyDrive')
    if _mydrive.exists():
        _peers = sorted(p.name for p in _mydrive.iterdir() if p.is_dir())[:12]
        print(f'[diag] /content/drive/MyDrive/ contains (up to 12): {_peers}')
    else:
        print('[diag] /content/drive/MyDrive/ not present — Drive mount failed or pending.')

# RESULTS_DIR / FIGURES_DIR default to DATA_ROOT/{results,figures} so they match
# the companion notebook's save location (critical — the companion writes
# step4_summary.json + features_df.pkl to DATA_ROOT/results/).
_default_results = DATA_ROOT / 'results' if DATA_ROOT.exists() else Path('results')
_default_figures = DATA_ROOT / 'figures' if DATA_ROOT.exists() else Path('figures')
RESULTS_DIR = Path(os.environ.get('SEAMLESS_RESULTS_DIR', str(_default_results)))
FIGURES_DIR = Path(os.environ.get('SEAMLESS_FIGURES_DIR', str(_default_figures)))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Add repo root to sys.path so `from experiments.signal_utils import ...` works
# in §2H sidecar cells (anchor_crqa_to_rupture_events, hmm_regime_segmentation).
if DATA_ROOT.exists() and str(DATA_ROOT) not in sys.path:
    sys.path.insert(0, str(DATA_ROOT))
_su_path = DATA_ROOT / 'experiments' / 'signal_utils.py'
if _su_path.exists():
    print(f'[run-ready] experiments module on sys.path: {_su_path}')
else:
    print(f'[warn] experiments/signal_utils.py not found at {_su_path} — §2H sidecar cells will skip.')

print(f'[run-ready] DATA_ROOT={DATA_ROOT}, RESULTS_DIR={RESULTS_DIR}, FIGURES_DIR={FIGURES_DIR}')



In [ ]:
# ============================================================================
# 1A. Core imports — with ewstools pin + inline fallback
# ============================================================================
# audit fix: ewstools can fail on Colab if numpy/scipy versions drift.
# Pin exact version; if import fails, inline fallback provides variance + AR(1)
# + skewness only (fold/Hopf spectra unavailable, gated downstream).

import numpy as np
import pandas as pd
import scipy.stats as _stats
from scipy.stats import kendalltau, spearmanr
from scipy.signal import welch
# Private aliases for compute_csd_indicators / _indicators_over_window
_welch_csd = welch
_ADF_AVAILABLE = False
try:
    from statsmodels.tsa.stattools import adfuller as _adfuller
    _ADF_AVAILABLE = True
except ImportError:
    def _adfuller(*args, **kwargs):  # type: ignore[misc]
        raise ImportError('statsmodels required for adfuller')
from pathlib import Path
import pickle
import json as _json
import warnings as _warnings

# ewstools (primary path) — version introspection is tolerant because some
# ewstools builds (including the one Colab installs by default) do not expose
# `__version__` as a module attribute. The canonical fallback uses
# `importlib.metadata.version` (Python >= 3.8, PEP 566). If both fail we
# print "unknown" — the ewstools functionality itself is unaffected.
EWSTOOLS_AVAILABLE = False
try:
    import ewstools
    EWSTOOLS_AVAILABLE = True
    _ews_ver = getattr(ewstools, "__version__", None)
    if _ews_ver is None:
        try:
            from importlib.metadata import version as _pkg_version
            _ews_ver = _pkg_version("ewstools")
        except Exception:
            _ews_ver = "unknown"
    print(f"[ok] ewstools {_ews_ver} loaded")
except ImportError:
    print("[warn] ewstools not available — falling back to inline variance / AR(1) / skew")
    print("       Install with: pip install 'ewstools>=2.1,<2.2'")
    print("       Fold-vs-Hopf spectra will be UNAVAILABLE under fallback.")

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# DTW (for arc clustering — unchanged from original notebook)
try:
    from dtaidistance import dtw
    DTW_AVAILABLE = True
except ImportError:
    DTW_AVAILABLE = False
    print("[warn] dtaidistance not available — arc clustering (§3) will be skipped")

#  self-heal: if kernel was restarted after Cell 3 (common when
# !pip install upgrades numpy/scipy), SEED lives only in the dead kernel.
# Re-define with the documented default so Cell 5 does not NameError.
if 'SEED' not in dir():
    import os as _os_seed
    SEED = int(_os_seed.environ.get('SEAMLESS_SEED', 42))
    _warnings.warn(
        f'SEED not found in kernel — re-seeding to {SEED}. Cell 3 likely '
        f'did not complete (kernel-restart trap after pip install). '
        f'Re-run Cell 3 if Drive mount / deps are also unavailable.',
        RuntimeWarning,
    )

print(f"\n[status] ewstools={EWSTOOLS_AVAILABLE}, dtaidistance={DTW_AVAILABLE}")
print(f"[status] SEED={SEED}, DATA_ROOT={DATA_ROOT}, RESULTS_DIR={RESULTS_DIR}")


In [ ]:
# ============================================================================
# 1B. Load companion notebook outputs (audit: replaces stale kuramoto_results.pkl)
# ============================================================================
# companion notebook produces three artifacts in RESULTS_DIR:
#   (a) step4_table1.csv   — per-model AUC summary
#   (b) step4_summary.json — primary AUC + d + result_label + achieved_power
#   (c) features_df.pkl    — per-dyad features (canonical_r, null_z, weights, T,
#                              condition, relationship, dyad_id)
# companion notebook also needs per-frame coupling trajectories for CSD. The `features_df` from
# companion notebook holds per-dyad SCALARS, not time series. For CSD we compute rolling CCA_1
# via the canonical directions cached at companion notebook (`dyad_cca_inputs` dict).
# If that cache is missing, we fall back to rolling Pearson on z-scored body
# motion energy (coarse but defensible proxy).

_required = ["RESULTS_DIR", "DATA_ROOT", "np", "pd"]
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise NameError(f"companion notebook Cell 1B requires {_missing} from Section 1A")

# Preferred: companion notebook artifacts
summary_path = RESULTS_DIR / "step4_summary.json"
features_path = RESULTS_DIR / "features_df.pkl"
legacy_path = RESULTS_DIR / "kuramoto_results.pkl"

nb1_summary = None
nb1_features = None

if summary_path.exists():
    with open(summary_path) as f:
        nb1_summary = _json.load(f)
    print(f"[ok] Loaded companion notebook summary: primary_model={nb1_summary.get('primary_model')}, "
          f"AUC={nb1_summary.get('primary_auc')}, label={nb1_summary.get('result_label')}")
else:
    print(f"[warn] {summary_path} not found — companion notebook has not been executed yet.")

if features_path.exists():
    try:
        nb1_features = pd.read_pickle(features_path)
        print(f"[ok] Loaded features_df: {len(nb1_features)} dyads, "
              f"columns: {list(nb1_features.columns)[:6]}...")
    except Exception as e:
        print(f"[warn] features_df.pkl load failed: {e}")
elif legacy_path.exists():
    _warnings.warn(
        f"{features_path} not found but legacy {legacy_path} exists — "
        "companion notebook output schema has changed since the original kuramoto pipeline. "
        "Re-run companion notebook to produce the current features_df.",
        RuntimeWarning,
    )
else:
    print(f"[warn] Neither {features_path} nor {legacy_path} found.")
    print(f"       companion notebook will run on synthetic demo data below.")

# Fallback: synthetic demo if companion notebook outputs unavailable
if nb1_features is None:
    print("\n[demo] Generating synthetic dyad signals for notebook smoke test.")
    _rng = np.random.default_rng(SEED)
    _T = 9000   # 5 minutes at 30 Hz
    _n_demo = 10
    demo_dyads = {}
    for i in range(_n_demo):
        coupling_strength = _rng.uniform(0.15, 0.55)
        t = np.arange(_T) / 30.0
        shared = np.sin(2 * np.pi * 0.3 * t) + 0.5 * np.sin(2 * np.pi * 0.7 * t)
        sig_a = coupling_strength * shared + (1 - coupling_strength) * _rng.normal(0, 1, _T)
        sig_b = coupling_strength * shared + (1 - coupling_strength) * _rng.normal(0, 1, _T)
        # Also build synthetic head_motion + gaze_div for rupture detection
        head_a = _rng.normal(0, 0.3, (_T, 3)).cumsum(axis=0)  # (T, 3) matches Seamless shape
        head_b = _rng.normal(0, 0.3, (_T, 3)).cumsum(axis=0)  # (T, 3)
        gaze_a = _rng.normal(0, 1, (_T, 2))
        gaze_b = _rng.normal(0, 1, (_T, 2))
        demo_dyads[f"demo_{i:03d}"] = {
            "cca_a": sig_a, "cca_b": sig_b,
            "head_a": head_a, "head_b": head_b,
            "gaze_a": gaze_a, "gaze_b": gaze_b,
            "relationship": "familiar" if i < 5 else "stranger",
            "interaction_type": "ipc_conversation",
        }
    dyad_streams = demo_dyads
    print(f"[demo] {len(dyad_streams)} synthetic dyads generated.")
else:
    # Real path: iterate companion notebook feature_df and load per-frame streams from NPZ
    dyad_streams = {}  # Will be populated in the extract loop below (cell 1C)
    print(f"[ok] Ready to build per-frame streams for {len(nb1_features)} real dyads.")


In [ ]:
# ============================================================================
# 1C. Build per-frame streams for real dyads (NPZ → CCA → dyad_streams)
# ============================================================================
# For each dyad in nb1_features we:
#   1. Load NPZ via signal_utils.load_dyad_from_manifest
#   2. Build multimodal CCA features (body + FAU + head) via
#      signal_utils.extract_dyad_cca_features
#   3. Fit CCA_1 via signal_utils.compute_cca_1(return_result=True) —
#      cca_result.cca_a_1 / cca_b_1 are the per-frame CCA_1(t) trajectories
#   4. Extract per-frame head_encodings + gaze_encodings for the
#      independence-preserving rupture signal
#   5. Pack into dyad_streams[interaction_key] dict
# Runtime budget: ~1-2 s per dyad. Default NB2_MAX_LOAD_DYADS=20 for a fast
# pilot pass; set the env var to 186 (or 0 for all) for the full H1 run
# (~3-6 min on Colab Pro).

if nb1_features is None:
    # Synthetic demo path already built dyad_streams in Cell 1B; nothing to do.
    print("[skip] Cell 1C — features_df missing, dyad_streams already has synthetic demo data.")
else:
    try:
        from experiments.signal_utils import (
            load_dyad_from_manifest,
            extract_dyad_cca_features,
            compute_cca_1,
        )
        _SU_CCA_AVAILABLE = True
    except (ImportError, ModuleNotFoundError) as _su_e:
        _SU_CCA_AVAILABLE = False
        print(f"[skip] Cell 1C — experiments.signal_utils not importable: {_su_e}")
        print("      Cell 4 prelude should have set sys.path; re-run Cell 4 or check DATA_ROOT.")
        dyad_streams = {}

    if _SU_CCA_AVAILABLE:
        # Resolve manifest + NPZ directory
        MANIFEST_PATH = DATA_ROOT / "pipeline_manifest.json"
        NPZ_DIR = DATA_ROOT / "npz"

        if not MANIFEST_PATH.exists():
            print(f"[warn] manifest not found: {MANIFEST_PATH}")
            print("      Run the companion data-pipeline notebook first to produce it.")
            dyad_streams = {}
        elif not NPZ_DIR.exists():
            print(f"[warn] NPZ directory not found: {NPZ_DIR}")
            print("      Run the companion data-pipeline notebook to download NPZ files.")
            dyad_streams = {}
        else:
            # Preload manifest once to amortise JSON parse cost across 186 dyads.
            with open(MANIFEST_PATH) as _mf:
                _preloaded_manifest = _json.load(_mf)

            # Default per-modality bandpasses — match the empirical PSD values
            # computed in the companion notebook Apr-19 run; used as a sensible
            # fallback if the companion notebook did not save its own PSD diagnostic.
            _modality_bandpass = {
                "body": (0.12, 1.29),
                "fau":  (0.23, 2.11),
                "head": (0.12, 1.29),
            }

            MAX_LOAD = int(os.environ.get("NB2_MAX_LOAD_DYADS", "0"))  # : default full run (all 186); set env var to e.g. "20" for pilot
            if MAX_LOAD == 0 or MAX_LOAD > len(nb1_features):
                MAX_LOAD = len(nb1_features)
            # v6: stratified sampling by relationship so the pilot sample
            # guarantees a familiar+stranger mix. The first-N slice was
            # session-block ordered (all 20 dyads came from one familiar-pair
            # session) and downstream TE / HMM comparisons had no stranger
            # group to compare against.
            _fam_ids = nb1_features[
                nb1_features["relationship"] == "familiar"
            ]["interaction_key"].tolist()
            _str_ids = nb1_features[
                nb1_features["relationship"] == "stranger"
            ]["interaction_key"].tolist()
            _rng_strat = np.random.default_rng(SEED)
            if MAX_LOAD >= len(nb1_features):
                # Full run: take all, no sampling needed
                dyad_ids = list(nb1_features["interaction_key"])
            else:
                _n_each = MAX_LOAD // 2
                _fam_sample = _rng_strat.choice(
                    _fam_ids, size=min(_n_each, len(_fam_ids)), replace=False
                ).tolist()
                _str_sample = _rng_strat.choice(
                    _str_ids, size=min(MAX_LOAD - len(_fam_sample), len(_str_ids)),
                    replace=False,
                ).tolist()
                dyad_ids = _fam_sample + _str_sample
                _rng_strat.shuffle(dyad_ids)
                dyad_ids = list(dyad_ids)
                print(
                    f"Stratified pilot sample: {len(_fam_sample)} familiar "
                    f"+ {len(_str_sample)} stranger (SEED-deterministic)"
                )

            dyad_streams = {}
            n_loaded = n_skipped_load = n_skipped_cca = n_err = 0

            print(f"Loading {len(dyad_ids)} dyads "
                  f"(default=full run on all {len(nb1_features)} dyads; "
                  f"set NB2_MAX_LOAD_DYADS=20 for pilot)...")

            for idx_d, ik in enumerate(dyad_ids):
                try:
                    dyad = load_dyad_from_manifest(
                        MANIFEST_PATH, NPZ_DIR, ik,
                        preloaded_manifest=_preloaded_manifest,
                    )
                    if dyad is None:
                        n_skipped_load += 1
                        continue

                    # Build CCA features + fit CCA_1
                    triple = extract_dyad_cca_features(
                        dyad,
                        modality_bandpass=_modality_bandpass,
                        fs=30.0,
                        use_intersected_valid_mask=True,
                        min_length=640,
                    )
                    if triple is None:
                        n_skipped_cca += 1
                        continue
                    features_a, features_b, _col_names = triple

                    cca_result = compute_cca_1(
                        features_a, features_b,
                        n_components=1, prewhiten=False, return_result=True,
                    )

                    # Extract per-frame head + gaze encodings for the rupture signal.
                    # Seamless NPZ keys use prefix "movement:" for these channels.
                    p0, p1 = dyad["participants"][0], dyad["participants"][1]
                    head_a = p0.get("movement:head_encodings")
                    head_b = p1.get("movement:head_encodings")
                    gaze_a = p0.get("movement:gaze_encodings")
                    gaze_b = p1.get("movement:gaze_encodings")
                    is_valid_a = p0.get("smplh:is_valid")
                    is_valid_b = p1.get("smplh:is_valid")

                    if head_a is None or head_b is None or gaze_a is None or gaze_b is None:
                        n_skipped_cca += 1
                        continue

                    # Relationship / condition labels from features_df
                    _row = nb1_features[nb1_features["interaction_key"] == ik].iloc[0]

                    dyad_streams[ik] = {
                        "_source": "real",  # v6: sentinel for Cell 13 weighted path
                        "cca_a": cca_result.cca_a_1,
                        "cca_b": cca_result.cca_b_1,
                        "weights_a": cca_result.weights_a,
                        "weights_b": cca_result.weights_b,
                        "head_a": head_a,
                        "head_b": head_b,
                        "gaze_a": gaze_a,
                        "gaze_b": gaze_b,
                        "is_valid_a": is_valid_a,
                        "is_valid_b": is_valid_b,
                        "relationship": str(_row.get("relationship", "unknown")),
                        "interaction_type": str(_row.get("condition", "naturalistic")),
                    }
                    n_loaded += 1
                except Exception as e:
                    n_err += 1
                    if n_err <= 3:
                        print(f"  [{idx_d+1}/{len(dyad_ids)}] {ik} error: {type(e).__name__}: {e}")

                if (idx_d + 1) % 25 == 0:
                    print(f"  [{idx_d+1}/{len(dyad_ids)}] loaded={n_loaded} "
                          f"skipped_load={n_skipped_load} skipped_cca={n_skipped_cca} errors={n_err}")

            print(f"\n✓ dyad_streams populated: {n_loaded}/{len(dyad_ids)} dyads "
                  f"(load-skipped={n_skipped_load}, cca-skipped={n_skipped_cca}, errors={n_err})")


## 2. Critical Slowing Down (CSD) — Theory and Implementation

### Physics of tipping points

Near a stable equilibrium of a noisy dynamical system, small perturbations decay at rate $\lambda_1 < 0$:

$$\delta x(t) = \delta x(0)\, e^{\lambda_1 t}$$

As the system approaches a bifurcation, $\lambda_1 \to 0^-$ and three signatures emerge in the residual fluctuations:

| Signature | Math | Intuition |
|---|---|---|
| Recovery time diverges | $\tau \to \infty$ | "Takes longer to bounce back" |
| AC(1) rises toward 1 | $\text{AC}(1) = e^{\lambda_1 \Delta t} \to 1$ | "Each state remembers its predecessor" |
| Variance rises | $\sigma^2 = \sigma^2_\text{noise} / (2|\lambda_1|)$ | "Fluctuations grow as restoring force weakens" |

The variance formula is the stationary solution of an Ornstein-Uhlenbeck process — the canonical mean-reverting model of fluctuations near equilibrium. Skewness appears as a third indicator in asymmetric potential wells (Dakos et al. 2012).

### Validated domains

| Domain | Reference | Observation |
|---|---|---|
| Ecology | Scheffer et al. 2009 (Nature) | Rising variance / AC(1) before lake regime shift |
| Climate | Dakos et al. 2008 (PNAS) | CSD in paleoclimate tipping points |
| Psychology | Wichers et al. 2016 (JAMA Psychiatry) | CSD in momentary affect predicts depression relapse |

**Our application.** CSD on the dyadic coupling signal `CCA_1(t)` — treating the interaction as a dynamical system near a behavioural bifurcation.

### Limits of CSD in short-series / high-dim settings

CSD has well-documented failure modes that constrain interpretation:

- **Meisel & Kuehn 2012** *Phys. Rev. E* — classical CSD indicators are unreliable in high-dimensional systems; multi-indicator criteria (≥ 2 of 3) are a partial remediation.
- **Lenton 2011** *Nature Climate Change* — reports false-positive rates in paleoclimate EWS studies; motivates the `[NO EFFECT vs NULL]` label in our verdict gate.
- **Ruths & Ruths 2014** *Science* — null results are the norm in network inference on short behavioural time series. Our `[UNDERPOWERED]` and `[NO EFFECT vs NULL]` paths are calibrated to this expectation.

### Detrending (Gaussian kernel)

Raw trends mimic CSD. We detrend residuals with a Gaussian kernel at `bandwidth_frac = 0.10` (Dakos 2012). Sensitivity sweep in §2C verifies the choice is on a stable plateau.

### ADF stationarity check

Rising AC(1) could also indicate a random walk (non-stationary). The Augmented Dickey-Fuller test distinguishes CSD (stationary, slow-recovering) from non-stationarity (no mean reversion). ADF is reported as a sidecar — it does not gate the verdict.


In [ ]:
# ============================================================================
# 2A. Signal helpers (refactored — audit)
# ============================================================================
# Removed: inline `rolling_csd_indicators` (replaced by ewstools wrapper in 2B)
# Removed: `detect_rupture_events` (legacy R(t)-drop; CIRCULAR with CCA_1).
# Kept:    `gaussian_detrend` (Dakos 2012 detrending).
# New:     `head_motion_energy_stream`, `gaze_divergence_stream`,
#          `compute_cross_modal_rupture` (independence-preserving composite),
#          `rolling_cca1_trajectory` (dyad coupling signal from canonical directions).

_required = ["np", "pd", "kendalltau", "SEED"]
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise NameError(f"companion notebook Cell 2A requires {_missing} from Section 1")


def gaussian_detrend(x, bandwidth_frac=0.10):
    """Gaussian-kernel detrending (Dakos et al. 2012 PLoS One 7:e41010,
    DOI:10.1371/journal.pone.0041010).

    bandwidth_frac: fraction of series length used as kernel sigma.
    Default 0.10 is Dakos recommendation; ewstools default is 0.20 (too
    coarse for 5-min behavioral series). See §2C for sensitivity sweep.
    """
    x = np.asarray(x, dtype=np.float64).ravel()
    n = len(x)
    sigma = max(3.0, n * float(bandwidth_frac))
    # Gaussian kernel convolution
    k = int(np.ceil(3 * sigma))
    t = np.arange(-k, k + 1)
    kernel = np.exp(-0.5 * (t / sigma) ** 2)
    kernel /= kernel.sum()
    # Pad with edge values to avoid boundary artifacts
    padded = np.pad(x, k, mode="edge")
    trend = np.convolve(padded, kernel, mode="valid")
    # Defensive: ensure trend length matches signal ( fix).
    if len(trend) != n:
        trend = np.resize(trend, n)
    return x - trend


def head_motion_energy_stream(head_encodings: np.ndarray) -> np.ndarray:
    """Per-frame head motion energy from (T, D) encoding matrix.

    D=3 for Seamless head_encodings. Returns (T,) magnitude of frame-to-frame
    differences. Prefixed with a zero to preserve length.

    Handles 1D input (scalar timeseries) with absolute differences — useful for
    synthetic smoke-test data or single-channel head summaries.

    NEURAL encoding caveat: Seamless `movement:head_encodings` are learned
    representations, not calibrated rotations. "Motion energy" here is
    change-of-representation, not physical head turn velocity.
    """
    if head_encodings is None or len(head_encodings) < 2:
        return np.array([])
    arr = np.asarray(head_encodings, dtype=np.float64)
    if arr.ndim == 1:
        # Scalar timeseries — use absolute differences (no cross-axis norm)
        diff = np.diff(arr)
        return np.concatenate([[0.0], np.abs(diff)])
    diff = np.diff(arr, axis=0)
    energy = np.linalg.norm(diff, axis=-1)
    return np.concatenate([[0.0], energy])


def gaze_divergence_stream(
    gaze_a: np.ndarray, gaze_b: np.ndarray
) -> np.ndarray:
    """Per-frame gaze divergence magnitude between two participants.

    For (T, D) encodings, returns (T,) ||gaze_a - gaze_b||. Handles
    1D scalar-gaze as absolute difference (synthetic fallback shape).
    """
    if gaze_a is None or gaze_b is None:
        return np.array([])
    a = np.asarray(gaze_a, dtype=np.float64)
    b = np.asarray(gaze_b, dtype=np.float64)
    T = min(len(a), len(b))
    if T == 0:
        return np.array([])
    if a.ndim == 1 or b.ndim == 1:
        # Scalar-gaze case (synthetic or single-channel): absolute difference
        return np.abs(a[:T] - b[:T])
    return np.linalg.norm(a[:T] - b[:T], axis=-1)


def compute_cross_modal_rupture(
    head_a, head_b, gaze_a, gaze_b,
    fps=30, percentile=75, sustain_s=0.75,
    is_valid_mask=None,
):
    """Independence-preserving rupture detection (audit 1 fix).

    CCA_1 in companion notebook consumes body motion + FAU + F0 prosody. The rupture signal
    MUST NOT overlap with those modalities, or the H1 claim becomes circular.
    This function uses ONLY head_encodings + gaze_encodings — fully disjoint
    from CCA_1 inputs.

    Composite score:
        R(t) = z(head_A_energy(t) + head_B_energy(t)) + z(gaze_divergence(t))

    Rupture event = R(t) ≥ `percentile`-th percentile AND sustained for
    ≥ `sustain_s` seconds.

    Returns dict:
        rupture_mask: (T,) bool
        rupture_events: list of (start_frame, end_frame) tuples
        R_t: (T,) composite score
    """
    head_e_a = head_motion_energy_stream(head_a)
    head_e_b = head_motion_energy_stream(head_b)
    gaze_div = gaze_divergence_stream(gaze_a, gaze_b)
    T = min(len(head_e_a), len(head_e_b), len(gaze_div))
    if T < fps * 10:   # need at least 10 s
        return {"rupture_mask": np.array([]), "rupture_events": [], "R_t": np.array([])}

    head_sum = head_e_a[:T] + head_e_b[:T]
    # Z-score each component
    def z(x):
        s = x.std()
        return (x - x.mean()) / (s + 1e-12)

    R_t = z(head_sum) + z(gaze_div[:T])

    # Apply is_valid mask (if provided) — treat invalid frames as non-rupture
    if is_valid_mask is not None and len(is_valid_mask) >= T:
        R_t = np.where(is_valid_mask[:T] > 0, R_t, R_t.min())

    thresh = float(np.percentile(R_t, percentile))
    above = R_t >= thresh
    # min_run = round(sustain_s * fps). At sustain_s=0.75, fps=30 → 23 frames
    # → 0.767 s effective floor (not exactly 0.75 s; the docstring range is the
    # target, the round() is the implementation).
    min_run = int(round(sustain_s * fps))

    # Extract contiguous runs where above holds for ≥ min_run frames
    rupture_mask = np.zeros_like(above)
    events = []
    start = None
    for i, a in enumerate(above):
        if a and start is None:
            start = i
        elif not a and start is not None:
            if i - start >= min_run:
                rupture_mask[start:i] = True
                events.append((start, i))
            start = None
    if start is not None and len(above) - start >= min_run:
        rupture_mask[start:] = True
        events.append((start, len(above)))

    return {"rupture_mask": rupture_mask, "rupture_events": events, "R_t": R_t,
            "threshold": thresh, "T_used": T}


def rolling_cca1_trajectory(sig_a, sig_b, weights_a, weights_b,
                             window_s=5.0, step_s=0.5, fps=30):
    """ CAVEAT: This uses FIXED canonical directions from
    the full-session CCA fit, not rolling-refit directions. For short
    pre-rupture windows, locally-optimal canonical directions may
    differ substantially from the global fit, which can under-estimate
    local coupling and inject spurious persistence structure into
    rolling Pearson. If the bias is rupture-time-correlated (ruptures
    are where local dynamics diverge most from session-average), the
    Kendall τ in the warning-window test is biased. future work
    remediation: rolling CCA refits on a subset for sensitivity.

    Compute CCA_1 time series by projecting signals through cached canonical
    directions, then rolling Pearson correlation over `window_s` windows.

    This is the companion notebook analog of the windowed Kuramoto R(t) from the original
    notebook. Cached canonical directions (`weights_a`, `weights_b`) come
    from the per-dyad CCA fit in companion notebook — loaded via dyad_cca_inputs.

    Returns (t_centers, cca1_rolling) where `cca1_rolling` is the Pearson
    correlation between projected sig_a and sig_b in each window.
    """
    sig_a = np.asarray(sig_a, dtype=np.float64)
    sig_b = np.asarray(sig_b, dtype=np.float64)
    T = min(sig_a.shape[0], sig_b.shape[0])

    # Project through canonical directions
    if weights_a is not None and sig_a.ndim == 2:
        xc = sig_a[:T] @ weights_a.ravel()
    else:
        xc = sig_a[:T].ravel() if sig_a.ndim == 1 else sig_a[:T, 0]
    if weights_b is not None and sig_b.ndim == 2:
        yc = sig_b[:T] @ weights_b.ravel()
    else:
        yc = sig_b[:T].ravel() if sig_b.ndim == 1 else sig_b[:T, 0]

    window = int(round(window_s * fps))
    step = int(round(step_s * fps))
    if T < window + 1:
        return np.array([]), np.array([])

    centers = []
    vals = []
    for start in range(0, T - window + 1, step):
        xw = xc[start : start + window]
        yw = yc[start : start + window]
        if xw.std() < 1e-9 or yw.std() < 1e-9:
            vals.append(np.nan)
        else:
            r = float(np.corrcoef(xw, yw)[0, 1])
            vals.append(r if np.isfinite(r) else np.nan)
        centers.append((start + window // 2) / fps)
    return np.array(centers), np.array(vals)


print("[ok] Cross-modal rupture + rolling CCA_1 helpers loaded.")
print("     Independence: rupture uses head + gaze only (disjoint from CCA_1).")
print("     Detrending: Dakos 2012 Gaussian-kernel, bandwidth_frac=0.10 default.")


[ok] Cross-modal rupture + rolling CCA_1 helpers loaded.
     Independence: rupture uses head + gaze only (disjoint from CCA_1).
     Detrending: Dakos 2012 Gaussian-kernel, bandwidth_frac=0.10 default.


In [ ]:
# ============================================================================
# 2B. ewstools wrapper with inline fallback (audit 5 fix)
# ============================================================================
# rolling window is a SEPARATE
# parameter from detrending bandwidth. bandwidth_frac controls the
# Gaussian kernel width in gaussian_detrend; ROLLING_WINDOW_FRAC
# controls the rolling window for variance / AR(1) / skew.
ROLLING_WINDOW_FRAC = 0.50   # Dakos 2012 + ewstools docs standard
MIN_WINDOW_SAMPLES = 60       # Unified min samples ( fix)

def compute_csd_indicators(ts, bandwidth_frac=0.10, min_samples=MIN_WINDOW_SAMPLES):
    """Compute CSD indicators on a 1-D time series with ewstools or fallback.

    Returns dict of {variance, ar1, skewness, fold_weight, hopf_weight,
    method}. fold_weight / hopf_weight are NaN under fallback (ewstools-only).

    Parameters
    ----------
    ts : 1-D array
        Detrended coupling signal (typically CCA_1(t) after gaussian_detrend).
    bandwidth_frac : float
        Fraction of series length used as kernel sigma for detrending.
        Dakos 2012 recommends 0.10 (default); ewstools default is 0.20.
    min_samples : int
        Floor for series length; return NaNs below this.
    """
    ts = np.asarray(ts, dtype=np.float64).ravel()
    ts = ts[np.isfinite(ts)]
    if len(ts) < min_samples:
        return {"variance": np.nan, "ar1": np.nan, "skewness": np.nan,
                "fold_weight": np.nan, "hopf_weight": np.nan,
                "spectral_reddening": np.nan, "adf_stat": np.nan, "adf_p": np.nan,
                "method": "insufficient_samples", "n_samples": len(ts)}

    detrended = gaussian_detrend(ts, bandwidth_frac=bandwidth_frac)

    if EWSTOOLS_AVAILABLE:
        try:
            # ewstools API: create TimeSeries, compute EWS, optional spectrum
            series = pd.Series(detrended)
            ts_obj = ewstools.TimeSeries(data=series, transition=None)
            ts_obj.compute_var(rolling_window=ROLLING_WINDOW_FRAC)
            ts_obj.compute_auto(rolling_window=ROLLING_WINDOW_FRAC, lag=1)
            ts_obj.compute_skew(rolling_window=ROLLING_WINDOW_FRAC)
            try:
                ts_obj.compute_spectrum(ham_length=40, ham_offset=0.5,
                                        pspec_roll_offset=20, w_cutoff=1)
                # Fit fold / Hopf spectra — AIC weights available as attributes
                fold_w = float(getattr(ts_obj, "aic_fold", np.nan))
                hopf_w = float(getattr(ts_obj, "aic_hopf", np.nan))
            except Exception:
                fold_w = np.nan
                hopf_w = np.nan
            # Take the LAST valid rolling value as the end-of-window indicator
            var_last = float(ts_obj.ews["variance"].dropna().iloc[-1]) \
                if "variance" in ts_obj.ews else np.nan
            ar1_last = float(ts_obj.ews["ac1"].dropna().iloc[-1]) \
                if "ac1" in ts_obj.ews else np.nan
            skew_last = float(ts_obj.ews["skew"].dropna().iloc[-1]) \
                if "skew" in ts_obj.ews else np.nan
            # Spectral reddening: low-to-high band power ratio (Bury, Bauch & Anand 2020
            # J. R. Soc. Interface 17:20200482, DOI:10.1098/rsif.2020.0482). Low band
            # 0-0.5 Hz captures CSD drift; high band 0.5-3 Hz is the active dyadic range.
            try:
                f_, psd_ = _welch_csd(detrended, fs=30.0, nperseg=min(256, len(detrended)))
                lo_ = float(psd_[(f_ > 0) & (f_ < 0.5)].mean()) if ((f_ > 0) & (f_ < 0.5)).any() else np.nan
                hi_ = float(psd_[(f_ >= 0.5) & (f_ < 3.0)].mean()) if ((f_ >= 0.5) & (f_ < 3.0)).any() else np.nan
                reddening = lo_ / (hi_ + 1e-12) if np.isfinite(lo_) and np.isfinite(hi_) else np.nan
            except Exception:
                reddening = np.nan
            # ADF stationarity test (theory anchor: Cell 7; statsmodels.tsa.stattools.adfuller
            # with default autolag=AIC, regression='c').
            if _ADF_AVAILABLE and len(detrended) >= 30:
                try:
                    _adf = _adfuller(detrended, maxlag=None, regression='c', autolag='AIC')
                    adf_stat, adf_p = float(_adf[0]), float(_adf[1])
                except Exception:
                    adf_stat, adf_p = np.nan, np.nan
            else:
                adf_stat, adf_p = np.nan, np.nan
            return {"variance": var_last, "ar1": ar1_last, "skewness": skew_last,
                    "fold_weight": fold_w, "hopf_weight": hopf_w,
                    "spectral_reddening": reddening, "adf_stat": adf_stat, "adf_p": adf_p,
                    "method": "ewstools", "n_samples": len(ts)}
        except Exception as e:
            _warnings.warn(f"ewstools failed, falling back to inline: {e}", RuntimeWarning)
            # fall through to inline

    # Inline fallback: variance + AR(1) + skewness + reddening + ADF
    var_ = float(np.var(detrended))
    if len(detrended) >= 2:
        ar1_ = float(np.corrcoef(detrended[:-1], detrended[1:])[0, 1])
    else:
        ar1_ = np.nan
    skew_ = float(_stats.skew(detrended))
    # Spectral reddening (Bury et al. 2020 framework)
    try:
        f_, psd_ = _welch_csd(detrended, fs=30.0, nperseg=min(256, len(detrended)))
        lo_ = float(psd_[(f_ > 0) & (f_ < 0.5)].mean()) if ((f_ > 0) & (f_ < 0.5)).any() else np.nan
        hi_ = float(psd_[(f_ >= 0.5) & (f_ < 3.0)].mean()) if ((f_ >= 0.5) & (f_ < 3.0)).any() else np.nan
        reddening = lo_ / (hi_ + 1e-12) if np.isfinite(lo_) and np.isfinite(hi_) else np.nan
    except Exception:
        reddening = np.nan
    # ADF stationarity test
    if _ADF_AVAILABLE and len(detrended) >= 30:
        try:
            _adf = _adfuller(detrended, maxlag=None, regression='c', autolag='AIC')
            adf_stat, adf_p = float(_adf[0]), float(_adf[1])
        except Exception:
            adf_stat, adf_p = np.nan, np.nan
    else:
        adf_stat, adf_p = np.nan, np.nan
    return {"variance": var_, "ar1": ar1_, "skewness": skew_,
            "fold_weight": np.nan, "hopf_weight": np.nan,
            "spectral_reddening": reddening, "adf_stat": adf_stat, "adf_p": adf_p,
            "method": "inline_fallback", "n_samples": len(ts)}


# Smoke test
_smoke_rng = np.random.default_rng(SEED)
_smoke_sig = _smoke_rng.normal(0, 1, 600) + np.linspace(0, 0.5, 600)
_smoke = compute_csd_indicators(_smoke_sig)
print(f"[smoke] CSD indicators on synthetic noise+drift: "
      f"variance={_smoke['variance']:.3f}, ar1={_smoke['ar1']:.3f}, "
      f"method={_smoke['method']}")


[smoke] CSD indicators on synthetic noise+drift: variance=1.059, ar1=0.105, method=ewstools


In [ ]:
# ============================================================================
# 2C. Bandwidth sensitivity sweep (audit 4 fix)
# ============================================================================
# ewstools default is 0.20 Gaussian bandwidth; Dakos 2012 recommends 0.10.
# We verify our chosen 0.10 is in a stable plateau, not an edge case.

_smoke_rng = np.random.default_rng(SEED)
_bw_test_sig = _smoke_rng.normal(0, 1, 1800) + np.linspace(0, 2.0, 1800)   # 1 min @ 30 Hz with drift
_bandwidths = [0.05, 0.10, 0.15, 0.20]
print("Bandwidth sensitivity sweep (synthetic CSD signal with known drift):")
print(f"  {'bandwidth_frac':<15s} {'variance':>10s} {'ar1':>10s} {'skewness':>10s}")
bw_results = []
for bw in _bandwidths:
    r = compute_csd_indicators(_bw_test_sig, bandwidth_frac=bw)
    bw_results.append((bw, r["variance"], r["ar1"], r["skewness"]))
    print(f"  {bw:<15.2f} {r['variance']:>10.3f} {r['ar1']:>10.3f} {r['skewness']:>10.3f}")

print("\nInterpretation: if variance/ar1 are stable across bandwidths (differ <20%),")
print("our 0.10 choice is in a plateau. If they differ sharply, report bandwidth-")
print("dependence as a sensitivity caveat in the H1 claim. Applied to real dyads:")
print(f"  (see cell 2D full CSD loop below.)")

# initialize None, set only on valid data
SENSITIVITY_PASSES = None
if len(bw_results) >= 2:
    vars_ = [r[1] for r in bw_results if np.isfinite(r[1]) and r[1] > 1e-6]
    if len(vars_) >= 2 and np.mean(vars_) > 1e-6:
        rel_range = (max(vars_) - min(vars_)) / np.mean(vars_)
        SENSITIVITY_PASSES = bool(rel_range < 0.20)
        print(f"\n  Variance range across bandwidths (relative): {rel_range:.3f}")
        print(f"  SENSITIVITY_PASSES = {SENSITIVITY_PASSES}  (threshold: <0.20)")
    else:
        print(f"\n  SENSITIVITY_PASSES = None (insufficient finite, nonzero variance samples)")


Bandwidth sensitivity sweep (synthetic CSD signal with known drift):
  bandwidth_frac    variance        ar1   skewness
  0.05                 1.040      0.004     -0.083
  0.10                 1.043      0.007     -0.084
  0.15                 1.043      0.007     -0.083
  0.20                 1.043      0.007     -0.081

Interpretation: if variance/ar1 are stable across bandwidths (differ <20%),
our 0.10 choice is in a plateau. If they differ sharply, report bandwidth-
dependence as a sensitivity caveat in the H1 claim. Applied to real dyads:
  (see cell 2D full CSD loop below.)

  Variance range across bandwidths (relative): 0.003
  SENSITIVITY_PASSES = True  (threshold: <0.20)


In [ ]:
# ============================================================================
# 2D. Config toggles + full-loop gate
# ============================================================================
# Sprint-demo defaults: RUN_FULL_CSD_LOOP=False, small shuffle counts.

RUN_FULL_CSD_LOOP = False    # Set True for full 311-dyad run (~30 min on Colab)
RESTRICT_TO_IPC = False       # v6: disabled — interaction_type lives in
                               # interactions.csv, not features_df. All 311
                               # naturalistic dyads accepted. Milestone-2: join
                               # interactions.csv if task-type filter needed.
MIN_IS_VALID = 0.5            # Drop dyads with mean is_valid < 0.5 (per data inventory)
BANDWIDTH_FRAC = 0.10         # Dakos 2012 recommendation
# Rupture-detector calibration () — grounded in dyadic-interaction
# literature rather than arbitrary tuning. The earlier 90th + 2.0 s setting
# produced 7 events on 186 dyads in earlier runs, yielding [UNDERPOWERED]
# verdicts mechanically. The parameter pair below targets ~3-6 events/dyad,
# above Politis & Romano 1994's ≥ 5-events-per-unit guideline for stable
# Kendall τ inference under circular block bootstrap:
#   - 75th percentile per-dyad — Fusaroli et al. 2014 *New Ideas in Psychology*
#     DOI:10.1016/j.newideapsych.2013.03.005 use ~75th percentile for dialog
#     CRQA rupture thresholding. Relaxes the hidden per-dyad 10% floor that
#     a 90th-percentile setting structurally imposes.
#   - 0.75 s sustain (23 frames @ 30 fps) — chosen to sit inside the 0.5-2.0 s
#     range typical of short behavioural-event detection. No single paper
#     specifies this exact duration for gaze+head composite z-scores; it is
#     a middle-ground choice between aggressive (0.5 s) and conservative
#     (2.0 s) settings. Ramseyer & Tschacher 2011's 5 s sustain is the
#     CONTINUOUS-body-sway convention and does not transfer to discrete
#     composite-score events. Dablander et al. 2026 PNAS (DOI:10.1073/pnas.2503493122)
#     is cited for the multi-modal ML-EWS framework generally, NOT for this
#     exact sustain parameter. Sensitivity to (percentile, sustain) is tracked
#     as a backlog item for post-ship sensitivity analysis.
RUPTURE_PERCENTILE = 75
RUPTURE_SUSTAIN_S = 0.75
# Horizons: h=30 dropped in  because WARNING_LEAD_S=30 makes the
# pre-window [t_r - 30, t_r - 30] zero-length, producing NaN by construction
# ( diagnostic cell flagged this explicitly).
WARNING_HORIZONS_S = [60, 90]   # multi-horizon robustness (Wichers 2016 supports 60-120 s)
WARNING_WINDOW_S = 60           # DEPRECATED alias () — downstream uses PRIMARY_HORIZON_S (Cell 2F) and WARNING_HORIZONS_S; kept only for back-compat with any external consumers.
WARNING_LEAD_S = 30             # exclude [t_r - 30 s, t_r]: pre-rupture trajectory 90 s-30 s
N_PARTNER_SHUFFLES = 50 if not RUN_FULL_CSD_LOOP else 500
N_TIME_SHUFFLES = 50 if not RUN_FULL_CSD_LOOP else 500

# Sample-size floor for H1 interpretation
MIN_N_EFF_DYADS = 100
MIN_N_EVENTS = 200

print("NB2 config:")
for k in ["RUN_FULL_CSD_LOOP", "RESTRICT_TO_IPC", "MIN_IS_VALID", "BANDWIDTH_FRAC",
          "RUPTURE_PERCENTILE", "RUPTURE_SUSTAIN_S", "WARNING_WINDOW_S", "WARNING_HORIZONS_S",
          "N_PARTNER_SHUFFLES", "N_TIME_SHUFFLES",
          "MIN_N_EFF_DYADS", "MIN_N_EVENTS"]:
    print(f"  {k} = {globals()[k]}")


In [ ]:
# ============================================================================
# 2E. Sample-size floor diagnostic + full CSD loop (audit 3 fix)
# ============================================================================
# Report event count histogram + N_eff + N_events BEFORE any H1 claim.

_required = ["dyad_streams", "compute_cross_modal_rupture",
             "rolling_cca1_trajectory", "compute_csd_indicators"]
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise NameError(f"companion notebook Cell 2E requires {_missing}")

# Apply is_valid + interaction_type filters
def _filter_dyads(stream_dict):
    kept = {}
    for ik, d in stream_dict.items():
        if RESTRICT_TO_IPC and d.get("interaction_type", "ipc_conversation") != "ipc_conversation":
            continue
        # Skip is_valid filter for demo data (no per-frame is_valid)
        iv_a = d.get("is_valid_a"); iv_b = d.get("is_valid_b")
        if iv_a is not None and iv_b is not None:
            if float(np.mean(iv_a)) < MIN_IS_VALID or float(np.mean(iv_b)) < MIN_IS_VALID:
                continue
        kept[ik] = d
    return kept

dyads_filtered = _filter_dyads(dyad_streams)
print(f"After filters: {len(dyads_filtered)}/{len(dyad_streams)} dyads kept "
      f"(RESTRICT_TO_IPC={RESTRICT_TO_IPC}, MIN_IS_VALID={MIN_IS_VALID})")

# Compute rupture events + CCA_1(t) per dyad
per_dyad_results = {}
event_counts = []
for ik, d in dyads_filtered.items():
    rupt = compute_cross_modal_rupture(
        d.get("head_a"), d.get("head_b"),
        d.get("gaze_a"), d.get("gaze_b"),
        fps=30, percentile=RUPTURE_PERCENTILE,
        sustain_s=RUPTURE_SUSTAIN_S,
    )
    # v6 both-paths: primary path is rolling Pearson on pre-projected scalars
    # (cca_a / cca_b from compute_cca_1.cca_a_1 / cca_b_1). Real dyads also get
    # a weighted re-projection sidecar via the cached canonical directions
    # (weights_a / weights_b) from Cell 7. Both stored in per_dyad_results for
    # diagnostic overlay in Cell 25 summary.
    if "cca_a" in d:
        centers, cca1_t = rolling_cca1_trajectory(
            d["cca_a"], d["cca_b"], weights_a=None, weights_b=None,
            window_s=5.0, step_s=0.5, fps=30,
        )
        if d.get("_source") == "real" and "weights_a" in d and "weights_b" in d:
            try:
                _centers_w, cca1_t_weighted = rolling_cca1_trajectory(
                    d["cca_a"], d["cca_b"],
                    weights_a=d["weights_a"], weights_b=d["weights_b"],
                    window_s=5.0, step_s=0.5, fps=30,
                )
            except Exception:
                cca1_t_weighted = np.array([])
        else:
            cca1_t_weighted = np.array([])
    else:
        centers, cca1_t = np.array([]), np.array([])
        cca1_t_weighted = np.array([])
    per_dyad_results[ik] = {
        "rupture": rupt,
        "centers": centers,
        "cca1_t": cca1_t,
        "cca1_t_weighted": cca1_t_weighted,  # v6: diagnostic sidecar
        "relationship": d.get("relationship"),
    }
    event_counts.append(len(rupt["rupture_events"]))

event_counts = np.array(event_counts)
n_eff_dyads = int((event_counts >= 1).sum())
n_events_total = int(event_counts.sum())
print(f"\nEvent count histogram:")
for lo, hi in [(0, 1), (1, 2), (2, 5), (5, 10), (10, 100)]:
    mask = (event_counts >= lo) & (event_counts < hi)
    print(f"  {lo:>3d}-{hi-1:<3d} events: {int(mask.sum()):>4d} dyads")
print(f"\n  n_eff_dyads (≥ 1 event) = {n_eff_dyads}")
print(f"  n_events_total            = {n_events_total}")
print(f"  MEETS H1 SAMPLE FLOOR:  n_eff_dyads ≥ {MIN_N_EFF_DYADS} "
      f"({'YES' if n_eff_dyads >= MIN_N_EFF_DYADS else 'NO'})  AND  "
      f"n_events ≥ {MIN_N_EVENTS} ({'YES' if n_events_total >= MIN_N_EVENTS else 'NO'})")

H1_SAMPLE_FLOOR_MET = (n_eff_dyads >= MIN_N_EFF_DYADS) and (n_events_total >= MIN_N_EVENTS)
if not H1_SAMPLE_FLOOR_MET:
    print(f"\n  ⚠️  H1 result will be labeled [UNDERPOWERED] regardless of τ statistics.")


After filters: 186/186 dyads kept (RESTRICT_TO_IPC=False, MIN_IS_VALID=0.5)

Event count histogram:
    0-0   events:    7 dyads
    1-1   events:   18 dyads
    2-4   events:   72 dyads
    5-9   events:   64 dyads
   10-99  events:   25 dyads

  n_eff_dyads (≥ 1 event) = 179
  n_events_total            = 997
  MEETS H1 SAMPLE FLOOR:  n_eff_dyads ≥ 100 (YES)  AND  n_events ≥ 200 (YES)


In [ ]:
# ============================================================================
# 2E.1 Event-sparsity audit ( diagnostic)
# ----------------------------------------------------------------------------
# Reports the attrition between raw rupture events and events that can produce
# a valid pre-window at each horizon.  shipped 6 events across 186 dyads
# at h=60 (floor=200) which surfaced as [UNDERPOWERED]; this diagnostic
# separates the contributing factors so intervention can target the right one:
#   (a) rupture detector threshold (RUPTURE_PERCENTILE / RUPTURE_SUSTAIN_S)
#   (b) session duration (ruptures too close to session start for pre-window)
#   (c) cca1_t NaN fraction (mask filter collapses below MIN_WINDOW_SAMPLES)
#   (d) h vs WARNING_LEAD_S collision (h=30 with WARNING_LEAD_S=30 → 0 length)
# ============================================================================

print(f"\n== Event-sparsity audit ==")

# (0) Overall event totals from Cell 2E
print(f"\n[0] Raw rupture events (Cell 2E):")
print(f"     n_eff_dyads (>=1 event) = {n_eff_dyads}")
print(f"     n_events_total          = {n_events_total}")

# (1) cca1_t NaN-fraction + session-duration distribution
_cca1_nan_frac = []
_session_dur_s = []
for ik, d in per_dyad_results.items():
    if len(d["cca1_t"]) == 0:
        continue
    _cca1_nan_frac.append(float((~np.isfinite(d["cca1_t"])).mean()))
    _session_dur_s.append(float(d["centers"][-1]) if len(d["centers"]) else 0.0)
_cca1_nan_frac = np.array(_cca1_nan_frac) if _cca1_nan_frac else np.array([np.nan])
_session_dur_s = np.array(_session_dur_s) if _session_dur_s else np.array([np.nan])

print(f"\n[1] cca1_t NaN fraction across {len(_cca1_nan_frac)} dyads:")
print(f"     median = {np.nanmedian(_cca1_nan_frac):.2%}, "
      f"max = {np.nanmax(_cca1_nan_frac):.2%}, "
      f">50% NaN: {int((_cca1_nan_frac > 0.5).sum())} dyads")

print(f"\n[2] Session duration distribution (seconds):")
for pct in [10, 25, 50, 75, 90]:
    v = float(np.nanpercentile(_session_dur_s, pct))
    print(f"     p{pct:>2d} = {v:.1f} s")

# (3) Pre-window validity projection per horizon
print(f"\n[3] Pre-window-validity attrition per horizon "
      f"(WARNING_LEAD_S={WARNING_LEAD_S}, MIN_WINDOW_SAMPLES={MIN_WINDOW_SAMPLES}):")
print(f"     {'horizon_s':>10} {'win_len_s':>10} {'min_samples_required':>22} {'zero_len?':>10}")
for h in WARNING_HORIZONS_S:
    win_s = h - WARNING_LEAD_S
    zero_len = win_s <= 0
    # At rolling-Pearson fs=2 Hz, min samples required = MIN_WINDOW_SAMPLES
    # (mask.sum() check happens on centers, not on raw frames).
    print(f"     {h:>10.0f} {win_s:>10.1f} {MIN_WINDOW_SAMPLES:>22d} {str(zero_len):>10}")

# (4) Per-horizon raw-event accounting: how many events satisfy t_start >= 0
print(f"\n[4] Raw events with t_start >= 0 per horizon:")
_raw_counts = {h: 0 for h in WARNING_HORIZONS_S}
for ik, d in per_dyad_results.items():
    # Defensive .get() — tolerate dyads whose
    # rupture dict lacks "rupture_events" (e.g., head/gaze all NaN).
    for (s, e) in d.get("rupture", {}).get("rupture_events", []):
        t_r = s / 30.0
        for h in WARNING_HORIZONS_S:
            if (t_r - h) >= 0 and (h - WARNING_LEAD_S) > 0:
                _raw_counts[h] += 1
for h in WARNING_HORIZONS_S:
    print(f"     h={h:>3.0f}: {_raw_counts[h]:>5d} raw events with valid pre-window geometry")

# (5) Intervention hints based on the numbers above
print(f"\n[5] Intervention hints (current config: "
      f"RUPTURE_PERCENTILE={RUPTURE_PERCENTILE}, "
      f"RUPTURE_SUSTAIN_S={RUPTURE_SUSTAIN_S}, "
      f"MIN_N_EVENTS={MIN_N_EVENTS}):")
if _raw_counts.get(60, 0) < MIN_N_EVENTS:
    print(f"     - h=60 raw count ({_raw_counts.get(60, 0)}) < MIN_N_EVENTS ({MIN_N_EVENTS}).")
    if (_cca1_nan_frac > 0.5).mean() > 0.2:
        print(f"       • >20% of dyads have >50% NaN in cca1_t; upstream CCA fit quality matters.")
    if _session_dur_s.size and np.nanmedian(_session_dur_s) < 90:
        print(f"       • Median session < 90 s; many events near session start rejected.")
    if n_events_total < MIN_N_EVENTS:
        print(f"       • Raw event count short — consider lowering RUPTURE_PERCENTILE below "
              f"{RUPTURE_PERCENTILE} or RUPTURE_SUSTAIN_S below {RUPTURE_SUSTAIN_S}.")
# Generic zero-length-horizon check (): warns for any horizon h where
# h - WARNING_LEAD_S <= 0. The h=30 hard-code was dead code after 
# dropped that horizon; this version is robust to future horizon-list changes.
for _h in WARNING_HORIZONS_S:
    if (_h - WARNING_LEAD_S) <= 0:
        print(f"     - h={_h} collapses to zero-length window (WARNING_LEAD_S={WARNING_LEAD_S}); "
              f"rate will be NaN by construction. Consider dropping h={_h} from WARNING_HORIZONS_S "
              f"or lowering WARNING_LEAD_S below {_h}.")


In [ ]:
# ============================================================================
# 2F. Multi-horizon warning window test with per-event Kendall τ
# ----------------------------------------------------------------------------
# Horizons grounded in Wichers et al. 2016 JAMA Psychiatry
# (DOI:10.1001/jamapsychiatry.2015.3079), which reports EWS detection windows
# of 60–120 s preceding affective transitions. We anchor the primary verdict
# at h=60 s and report h=90 s as a robustness sidecar. h=30 s was dropped in  because
# it collapses to a zero-length pre-window when WARNING_LEAD_S >= 30 s
# ( diagnostic cell 2E.1 confirmed this).
# ============================================================================

def _kendall_tau_trend(signal_slice):
    """Kendall τ for monotonic trend over a 1-D series."""
    signal_slice = signal_slice[np.isfinite(signal_slice)]
    if len(signal_slice) < 10:
        return np.nan, np.nan
    tau, p = kendalltau(np.arange(len(signal_slice)), signal_slice)
    return float(tau), float(p)


def _indicators_over_window(cca1_t, centers, t_start, t_end, bandwidth_frac):
    """Return dict of rolling variance / AR(1) / skewness trajectories plus
    single-valued reddening and ADF p-value over [t_start, t_end].

    Skewness grounded in Dakos et al. 2012 PLOS ONE (DOI:10.1371/journal.pone.0041010)
    as a complementary CSD indicator alongside variance and AR(1).
    Reddening computed via Welch PSD low 0–0.25 Hz / high 0.25–1.0 Hz ratio
    (Bury, Bauch & Anand 2020 framework, DOI:10.1098/rsif.2020.0482). Bands
    rescaled from the paper's 0.5 / 3 Hz defaults: cca1_t is the rolling-Pearson
    output of rolling_cca1_trajectory(step_s=0.5) → effective rate 2 Hz,
    Nyquist 1 Hz, so the 0.5–3 Hz band would be mostly above Nyquist and return
    a meaningless value.
    ADF test via statsmodels adfuller(regression='c', autolag='AIC').
    """
    if len(centers) == 0:
        return None
    mask = (centers >= t_start) & (centers <= t_end)
    if mask.sum() < MIN_WINDOW_SAMPLES:
        return None
    sub = cca1_t[mask]
    if not np.all(np.isfinite(sub)):
        sub = sub[np.isfinite(sub)]
        if len(sub) < MIN_WINDOW_SAMPLES:
            return None
    # Rolling window within the sub-trajectory (10-sample window)
    w = 10
    ser = pd.Series(sub)
    var_traj = ser.rolling(w).var().dropna().values
    skew_traj = ser.rolling(w).skew().dropna().values
    ar1_traj = np.array([
        float(np.corrcoef(sub[i:i+w][:-1], sub[i:i+w][1:])[0, 1])
        if sub[i:i+w].std() > 0 else np.nan
        for i in range(len(sub) - w + 1)
    ])

    # Single-valued scalars over the full sub-window.
    # fs=2.0 matches rolling-Pearson step_s=0.5; bands kept inside Nyquist=1 Hz.
    reddening_scalar = np.nan
    try:
        # nperseg=min(32, len(sub)): at 2 Hz this gives df = 2/32 = 0.0625 Hz,
        # so the 0.25 Hz band boundary sits cleanly at bin 4. Stable across all
        # valid horizons (60-s pre-window = 60 samples, 90-s = 120 samples).
        f, p = _welch_csd(sub, fs=2.0, nperseg=min(32, len(sub)))
        # Exclude DC bin (f=0): incomplete-detrending residuals put non-zero
        # mean into bin 0 which inflates low-band power and biases the
        # reddening ratio upward.
        low_band = (f > 0.0) & (f < 0.25)
        high_band = (f >= 0.25) & (f <= 1.0)
        # np.trapezoid (NumPy 2.0 replacement for np.trapz — signatures match)
        low_power = float(np.trapezoid(p[low_band], f[low_band])) if low_band.any() else np.nan
        high_power = float(np.trapezoid(p[high_band], f[high_band])) if high_band.any() else np.nan
        if np.isfinite(low_power) and np.isfinite(high_power) and high_power > 0:
            reddening_scalar = low_power / high_power
    except Exception:
        reddening_scalar = np.nan

    adf_p_scalar = np.nan
    if _ADF_AVAILABLE:
        try:
            adf_res = _adfuller(sub, maxlag=None, regression='c', autolag='AIC')
            adf_p_scalar = float(adf_res[1])
        except Exception:
            adf_p_scalar = np.nan

    return {
        "variance_traj": var_traj,
        "ar1_traj": ar1_traj,
        "skew_traj": skew_traj,
        "reddening_scalar": float(reddening_scalar) if np.isfinite(reddening_scalar) else np.nan,
        "adf_p_scalar": float(adf_p_scalar) if np.isfinite(adf_p_scalar) else np.nan,
    }


# Compute observed Kendall τ per event × indicator × horizon
PRIMARY_HORIZON_S = 60
per_event_results = []
for ik, d in per_dyad_results.items():
    if len(d["centers"]) == 0:
        continue
    T_total = float(d["centers"][-1]) if len(d["centers"]) else 0.0
    for (s, e) in d["rupture"]["rupture_events"]:
        t_r = s / 30.0    # rupture start time in seconds
        row = {
            "dyad_id": ik,
            "t_r": t_r,
            "relationship": d.get("relationship"),
        }
        reddening_primary = np.nan
        adf_p_primary = np.nan
        any_horizon_ok = False
        for h in WARNING_HORIZONS_S:
            t_start = t_r - h
            t_end = t_r - WARNING_LEAD_S
            col_var = f"tau_var_{h}"
            col_ar1 = f"tau_ar1_{h}"
            col_skew = f"tau_skew_{h}"
            if t_start < 0 or t_end <= t_start:
                row[col_var] = np.nan
                row[col_ar1] = np.nan
                row[col_skew] = np.nan
                continue
            ind = _indicators_over_window(
                d["cca1_t"], d["centers"], t_start, t_end, BANDWIDTH_FRAC
            )
            if ind is None:
                row[col_var] = np.nan
                row[col_ar1] = np.nan
                row[col_skew] = np.nan
                continue
            any_horizon_ok = True
            tau_var, _ = _kendall_tau_trend(ind["variance_traj"])
            tau_ar1, _ = _kendall_tau_trend(ind["ar1_traj"])
            tau_skew, _ = _kendall_tau_trend(ind["skew_traj"])
            row[col_var] = tau_var
            row[col_ar1] = tau_ar1
            row[col_skew] = tau_skew
            if h == PRIMARY_HORIZON_S:
                reddening_primary = ind["reddening_scalar"]
                adf_p_primary = ind["adf_p_scalar"]
        row["reddening_60"] = reddening_primary
        row["adf_p_60"] = adf_p_primary
        if any_horizon_ok:
            per_event_results.append(row)

events_df = pd.DataFrame(per_event_results)
print(f"Computed Kendall τ for {len(events_df)} pre-rupture warning windows.")


In [ ]:
# ============================================================================
# 2G. Empirical null via circular (moving-blocks) block bootstrap (Kunsch 1989 family)
# ----------------------------------------------------------------------------
# Replaces the prior uniform rupture-time shuffle. The uniform shuffle is
# under-conservative for non-stationary CCA_1(t) (drift with session time
# inflates Type I). Circular block bootstrap preserves short-range temporal
# dependence while destroying long-range structure — the appropriate null for
# CSD indicators on autocorrelated series. Reference: Politis & Romano 1994
# (DOI:10.1080/01621459.1994.10476870). Multi-horizon sweep follows Wichers
# et al. 2016 JAMA Psychiatry (DOI:10.1001/jamapsychiatry.2015.3079).
# ============================================================================

def _circ_block_shuffle(cca1, rng, block_s=5.0, fs=2.0):
    """Circular (moving-blocks) block bootstrap — Kunsch 1989 / Liu & Singh 1992
    family; cf. Politis & Romano 1994 stationary bootstrap (DOI:10.1080/01621459.1994.10476870).
    This implementation uses fixed block length b; P&R is the canonical
    reference for the block-bootstrap family.

    Defaults: block_s=5.0, fs=2.0 → block = 10 samples, matching the
    rolling-Pearson output (rolling_cca1_trajectory step_s=0.5 → 2 Hz). For
    typical 300–600 sample series at 2 Hz this satisfies the n >= 3 * b
    guideline without clamping. Minimum block length floor of 10 samples
    preserves short-range autocorrelation; clamps to len(cca1) // 3 for very
    short series and emits one warning.
    """
    b = max(int(block_s * fs), 10)
    n = len(cca1)
    if n < 3 * b:
        b_clamped = max(n // 3, 1)
        if b_clamped < b:
            _warnings.warn(
                f"_circ_block_shuffle: series length {n} < 3*block; "
                f"clamping b from {b} to {b_clamped}.",
                RuntimeWarning,
                stacklevel=2,
            )
            b = b_clamped
    if b <= 0 or n == 0:
        return np.asarray(cca1).copy()
    starts = rng.integers(0, n, size=int(np.ceil(n / b)))
    return np.concatenate([np.roll(cca1, -int(s))[:b] for s in starts])[:n]


rng_null = np.random.default_rng(SEED + 1)
null_taus = {
    f"tau_{kind}_{h}": []
    for kind in ("var", "ar1", "skew")
    for h in WARNING_HORIZONS_S
}
# Sidecar: uniform-shuffle null at h=60 for the one-time sanity comparison
null_uniform_h60 = {"tau_var": [], "tau_ar1": [], "tau_skew": []}

for ik, d in per_dyad_results.items():
    if len(d["centers"]) == 0 or len(d["rupture"]["rupture_events"]) == 0:
        continue
    T_total = float(d["centers"][-1]) if len(d["centers"]) else 0.0
    # Need a valid uniform range [max_horizon, T_total - 5] for the circular-
    # block fake rupture draw below — guard against T_total in (max_horizon,
    # max_horizon + 5] short-session edge case.
    if T_total - 5 <= max(WARNING_HORIZONS_S):
        continue
    cca1_real = np.asarray(d["cca1_t"])
    centers_real = np.asarray(d["centers"])
    min_t_uniform = PRIMARY_HORIZON_S
    max_t_uniform = T_total - 5
    for _ in range(N_TIME_SHUFFLES):
        # (a) Circular block bootstrap of cca1 — primary null
        cca1_shuf = _circ_block_shuffle(cca1_real, rng_null, block_s=5.0, fs=2.0)
        t_r_fake = rng_null.uniform(max(WARNING_HORIZONS_S), T_total - 5)
        for h in WARNING_HORIZONS_S:
            t_start = t_r_fake - h
            t_end = t_r_fake - WARNING_LEAD_S
            if t_start < 0 or t_end <= t_start:
                continue
            ind = _indicators_over_window(
                cca1_shuf, centers_real, t_start, t_end, BANDWIDTH_FRAC
            )
            if ind is None:
                continue
            tau_var, _ = _kendall_tau_trend(ind["variance_traj"])
            tau_ar1, _ = _kendall_tau_trend(ind["ar1_traj"])
            tau_skew, _ = _kendall_tau_trend(ind["skew_traj"])
            if np.isfinite(tau_var):
                null_taus[f"tau_var_{h}"].append(tau_var)
            if np.isfinite(tau_ar1):
                null_taus[f"tau_ar1_{h}"].append(tau_ar1)
            if np.isfinite(tau_skew):
                null_taus[f"tau_skew_{h}"].append(tau_skew)
        # (b) Uniform rupture-time shuffle on *real* series at h=60 only (sanity)
        if max_t_uniform > min_t_uniform:
            t_r_uni = rng_null.uniform(min_t_uniform, max_t_uniform)
            t_start_u = t_r_uni - PRIMARY_HORIZON_S
            t_end_u = t_r_uni - WARNING_LEAD_S
            if t_start_u >= 0 and t_end_u > t_start_u:
                ind_u = _indicators_over_window(
                    cca1_real, centers_real, t_start_u, t_end_u, BANDWIDTH_FRAC
                )
                if ind_u is not None:
                    tv_u, _ = _kendall_tau_trend(ind_u["variance_traj"])
                    ta_u, _ = _kendall_tau_trend(ind_u["ar1_traj"])
                    ts_u, _ = _kendall_tau_trend(ind_u["skew_traj"])
                    if np.isfinite(tv_u):
                        null_uniform_h60["tau_var"].append(tv_u)
                    if np.isfinite(ta_u):
                        null_uniform_h60["tau_ar1"].append(ta_u)
                    if np.isfinite(ts_u):
                        null_uniform_h60["tau_skew"].append(ts_u)

print(
    "Null distribution sizes (circular block bootstrap):"
    + "".join(
        f"\n  h={h}: var={len(null_taus[f'tau_var_{h}'])}, "
        f"ar1={len(null_taus[f'tau_ar1_{h}'])}, "
        f"skew={len(null_taus[f'tau_skew_{h}'])}"
        for h in WARNING_HORIZONS_S
    )
)

# Per-event success: observed τ > 95th pct of null (per indicator × horizon)
min_null_n = 50
have_primary_null = (
    len(null_taus[f"tau_var_{PRIMARY_HORIZON_S}"]) >= min_null_n
    and len(null_taus[f"tau_ar1_{PRIMARY_HORIZON_S}"]) >= min_null_n
    and len(null_taus[f"tau_skew_{PRIMARY_HORIZON_S}"]) >= min_null_n
)

if len(events_df) and have_primary_null:
    cutoffs = {}
    for h in WARNING_HORIZONS_S:
        for kind in ("var", "ar1", "skew"):
            key = f"tau_{kind}_{h}"
            arr = null_taus[key]
            cutoffs[f"{kind}_cutoff_{h}"] = (
                float(np.percentile(arr, 95)) if len(arr) >= min_null_n else np.nan
            )

    # Phipson & Smyth 2010 (Stat. Appl. Genet. Mol. Biol. 9:39,
    # DOI:10.2202/1544-6115.1585) continuity-corrected one-sided permutation
    # p-values: p = (n_exceed + 1) / (n_shuffles + 1). At finite n, strict
    # `obs > np.percentile(null, 95)` is anti-conservative (effective α ≈ 0.059
    # at n=50); PS gives exactly α-calibrated rejection. Code-reviewer audit
    # C2, .
    def _ps_p_onesided(obs, null_vals):
        if not np.isfinite(obs) or len(null_vals) == 0:
            return np.nan
        arr = np.asarray(null_vals, dtype=float)
        arr = arr[np.isfinite(arr)]
        if len(arr) == 0:
            return np.nan
        return (int((arr >= obs).sum()) + 1) / (len(arr) + 1)

    PS_ALPHA = 0.05  # per-indicator one-sided test
    for h in WARNING_HORIZONS_S:
        for kind in ("var", "ar1", "skew"):
            tau_key = f"tau_{kind}_{h}"
            events_df[f"{kind}_p_{h}"] = events_df[tau_key].apply(
                lambda obs, k=tau_key: _ps_p_onesided(obs, null_taus[k])
            )
            events_df[f"{kind}_success_{h}"] = events_df[f"{kind}_p_{h}"] <= PS_ALPHA
        # Legacy either-of-two (var OR ar1) back-compat
        events_df[f"either_success_{h}"] = (
            events_df[f"var_success_{h}"] | events_df[f"ar1_success_{h}"]
        )
        # Primary rule: >= 2 of 3 indicators reject null at horizon h
        succ_sum = (
            events_df[f"var_success_{h}"].astype(int)
            + events_df[f"ar1_success_{h}"].astype(int)
            + events_df[f"skew_success_{h}"].astype(int)
        )
        events_df[f"two_of_three_success_{h}"] = succ_sum >= 2

    # Horizon-stratified rates on valid-window-only subsets. A NaN tau at
    # h=90 means the dyad was too short for the 90s pre-window — excluding
    # those rows avoids censoring the h=90 rate downward.
    rate_2of3_by_horizon = {
        int(h): float(
            events_df.loc[events_df[f"tau_var_{h}"].notna(),
                          f"two_of_three_success_{h}"].mean()
        ) if events_df[f"tau_var_{h}"].notna().any() else float("nan")
        for h in WARNING_HORIZONS_S
    }
    n_events_by_horizon = {
        int(h): int(events_df[f"tau_var_{h}"].notna().sum())
        for h in WARNING_HORIZONS_S
    }
    rate_2of3_60 = rate_2of3_by_horizon[PRIMARY_HORIZON_S]
    # Filter NaN-tau rows so the legacy rate is computed on the same
    # valid-window denominator as rate_2of3.
    _valid_60 = events_df[f"tau_var_{PRIMARY_HORIZON_S}"].notna()
    rate_either_60 = (
        float(events_df.loc[_valid_60, f"either_success_{PRIMARY_HORIZON_S}"].mean())
        if _valid_60.any() else float("nan")
    )

    print(
        f"\nCircular-block null cutoffs @ h={PRIMARY_HORIZON_S}: "
        f"var τ > {cutoffs[f'var_cutoff_{PRIMARY_HORIZON_S}']:.3f}, "
        f"ar1 τ > {cutoffs[f'ar1_cutoff_{PRIMARY_HORIZON_S}']:.3f}, "
        f"skew τ > {cutoffs[f'skew_cutoff_{PRIMARY_HORIZON_S}']:.3f}"
    )
    u_v = (
        float(np.percentile(null_uniform_h60["tau_var"], 95))
        if len(null_uniform_h60["tau_var"]) >= min_null_n else np.nan
    )
    u_a = (
        float(np.percentile(null_uniform_h60["tau_ar1"], 95))
        if len(null_uniform_h60["tau_ar1"]) >= min_null_n else np.nan
    )
    u_s = (
        float(np.percentile(null_uniform_h60["tau_skew"], 95))
        if len(null_uniform_h60["tau_skew"]) >= min_null_n else np.nan
    )
    print(
        f"  [sanity] Uniform-shuffle null cutoffs @ h={PRIMARY_HORIZON_S}: "
        f"var τ > {u_v:.3f}, ar1 τ > {u_a:.3f}, skew τ > {u_s:.3f} "
        f"(circular-block used as primary)"
    )
    print(
        f"\nSuccess rate (2 of 3 indicators, PRIMARY) @ h=60: "
        f"{rate_2of3_60:.2%} of {len(events_df)} events"
    )
    print(f"Success rate (either var/ar1, legacy) @ h=60: {rate_either_60:.2%}")
    print(
        "Horizon robustness — rate_2of3: "
        + ", ".join(f"h={h}: {rate_2of3_by_horizon[h]:.2%}" for h in WARNING_HORIZONS_S)
    )

    # [Reviewer audit] Effective-events floor (stricter than n_events_total):
    # counts τ-computable events only. Window-length filtering in
    # _indicators_over_window can drop 30-40% of detected ruptures, so
    # n_events_total can satisfy MIN_N_EVENTS while len(events_df) does not.
    n_events_effective = len(events_df)
    H1_EVENTS_EFFECTIVE_FLOOR_MET = n_events_effective >= MIN_N_EVENTS
    print(
        f"  Effective (τ-computable) events: {n_events_effective} "
        f"(floor {MIN_N_EVENTS}: {'YES' if H1_EVENTS_EFFECTIVE_FLOOR_MET else 'NO'})"
    )

    # Phipson-Smyth continuity for small-N permutation p-values: (b+1)/(n+1).
    # With N_TIME_SHUFFLES=50 the correction is ~2%; with 500 it's ~0.2%.

    # H1 verdict (anchored on 2-of-3 rule at h=60)
    if not H1_SAMPLE_FLOOR_MET or not H1_EVENTS_EFFECTIVE_FLOOR_MET:
        H1_LABEL = "[UNDERPOWERED]"
    elif rate_2of3_60 >= 0.70:
        H1_LABEL = "[EMPIRICAL]"
    elif rate_2of3_60 >= 0.50:
        H1_LABEL = "[EMPIRICAL but weak]"
    else:
        H1_LABEL = "[NO EFFECT vs NULL]"
    print(f"\nH1 verdict: {H1_LABEL}")

    # Save
    out = {
        "h1_label": H1_LABEL,
        "rate_either_60": rate_either_60,
        "rate_2of3_60": rate_2of3_60,
        "rate_2of3_by_horizon": rate_2of3_by_horizon,
        "n_events_by_horizon": n_events_by_horizon,
        "var_cutoff_60": cutoffs[f"var_cutoff_{PRIMARY_HORIZON_S}"],
        "ar1_cutoff_60": cutoffs[f"ar1_cutoff_{PRIMARY_HORIZON_S}"],
        "skew_cutoff_60": cutoffs[f"skew_cutoff_{PRIMARY_HORIZON_S}"],
        "uniform_var_cutoff_60_sanity": u_v,
        "uniform_ar1_cutoff_60_sanity": u_a,
        "uniform_skew_cutoff_60_sanity": u_s,
        "n_events_effective": n_events_effective,
        "H1_SAMPLE_FLOOR_MET": bool(H1_SAMPLE_FLOOR_MET),
        "H1_EVENTS_EFFECTIVE_FLOOR_MET": bool(H1_EVENTS_EFFECTIVE_FLOOR_MET),
        "null_method": "circular_moving_blocks_bootstrap_cf_politis_romano_1994",
        "warning_horizons_s": list(WARNING_HORIZONS_S),
        "primary_horizon_s": PRIMARY_HORIZON_S,
    }
    results_path = RESULTS_DIR / "h1_verdict_summary.json"
    with open(results_path, "w") as f:
        _json.dump(out, f, indent=2)
    print(f"  Saved: {results_path}")
else:
    print("\n[skip] Insufficient data for H1 verdict. Re-run with real dyads.")


In [ ]:
# ============================================================================
# 2H. Transfer-entropy directionality sidecar (Schreiber 2000)
# ----------------------------------------------------------------------------
# Schreiber 2000 Phys. Rev. Lett. 85:461 (DOI:10.1103/PhysRevLett.85.461).
# Binned-histogram implementation — no new package dependency. Measures
# directional information flow TE(X→Y) = H(Y_{t+k} | Y_t) − H(Y_{t+k} | Y_t, X_t)
# between the two participants' per-frame CCA_1 trajectories at lags k ∈
# {1, 3, 5, 10} frames (at 30 Hz → 33 ms, 100 ms, 167 ms, 333 ms). Asymmetry
# (TE_A→B − TE_B→A) / (TE_A→B + TE_B→A) indicates leader-follower structure.
# Richardson et al. 2015 (ref 16) validated this approach on dyadic finger-
# tapping; the present cell ports it to CCA_1(t) on interactional dyads.
# ============================================================================

def _binned_te(x, y, lag=1, bins=8):
    """Binned-histogram transfer entropy TE(X → Y) at a given lag.

    TE(X → Y) = H(Y_{t+lag} | Y_t) − H(Y_{t+lag} | Y_t, X_t)
             = Σ p(y_{t+lag}, y_t, x_t) log[ p(y_{t+lag} | y_t, x_t) / p(y_{t+lag} | y_t) ]

    Computes via three joint histograms (3-D, 2-D, 2-D, 1-D) and the chain
    rule for conditional entropy. Returns TE in nats.
    """
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()
    n = min(len(x), len(y)) - lag
    if n < 50:
        return np.nan
    x_now = x[:n]
    y_now = y[:n]
    y_fut = y[lag:lag + n]
    # Discretise via equal-frequency bins (quantile binning — robust to scale)
    def _bin(a):
        qs = np.linspace(0, 1, bins + 1)
        edges = np.quantile(a, qs)
        edges[0] -= 1e-9; edges[-1] += 1e-9
        idx = np.digitize(a, edges[1:-1], right=False)
        return idx.astype(np.int64)
    xb = _bin(x_now)
    yb = _bin(y_now)
    yfb = _bin(y_fut)
    # Joint p(y_fut, y_now, x_now)
    try:
        p_3d, _ = np.histogramdd(np.stack([yfb, yb, xb], axis=1),
                                 bins=(bins, bins, bins))
    except Exception:
        return np.nan
    p_3d = p_3d / max(p_3d.sum(), 1e-12)
    # Marginals
    p_yny = p_3d.sum(axis=2)   # p(y_fut, y_now)
    p_yx = p_3d.sum(axis=0)    # p(y_now, x_now)
    p_y = p_3d.sum(axis=(0, 2))  # p(y_now)
    # Entropy helper (Shannon, base e)
    def _h(p):
        p_safe = p[p > 0]
        return float(-(p_safe * np.log(p_safe)).sum())
    # TE = H(Y_fut | Y_now) − H(Y_fut | Y_now, X_now)
    #    = [H(Y_fut, Y_now) − H(Y_now)] − [H(Y_fut, Y_now, X_now) − H(Y_now, X_now)]
    h_3d = _h(p_3d)
    h_yny = _h(p_yny)
    h_yx = _h(p_yx)
    h_y = _h(p_y)
    te = (h_yny - h_y) - (h_3d - h_yx)
    return float(te)


if "dyad_streams" in globals() and dyad_streams:
    te_lags = [1, 3, 5, 10]
    te_records = []
    for ik, st in dyad_streams.items():
        cca_a = np.asarray(st.get("cca_a", []))
        cca_b = np.asarray(st.get("cca_b", []))
        if len(cca_a) < 120 or len(cca_b) < 120:
            continue
        for lag in te_lags:
            try:
                te_ab = _binned_te(cca_a, cca_b, lag=lag, bins=8)
                te_ba = _binned_te(cca_b, cca_a, lag=lag, bins=8)
                denom = (te_ab + te_ba) if np.isfinite(te_ab + te_ba) and (te_ab + te_ba) > 0 else np.nan
                asym = (te_ab - te_ba) / denom if np.isfinite(denom) else np.nan
                te_records.append({
                    "dyad_id": ik,
                    "lag_frames": lag,
                    "TE_A_to_B": te_ab,
                    "TE_B_to_A": te_ba,
                    "asymmetry": asym,
                    "relationship": st.get("relationship", "unknown"),
                })
            except Exception:
                continue
    dyad_te_df = pd.DataFrame(te_records)
    if len(dyad_te_df):
        print(f"Transfer entropy computed for {dyad_te_df['dyad_id'].nunique()} dyads × "
              f"{len(te_lags)} lags = {len(dyad_te_df)} rows.")
        # Mean asymmetry by relationship group (at primary lag=3)
        _primary_lag_te = 3
        _sub = dyad_te_df[dyad_te_df["lag_frames"] == _primary_lag_te]
        if len(_sub):
            by_rel = _sub.groupby("relationship")["asymmetry"].agg(["mean", "std", "count"])
            print(f"\nTE asymmetry by relationship (lag={_primary_lag_te} frames, ~100 ms):")
            print(by_rel.to_string())
    else:
        print("[skip] TE directionality — no dyads with sufficient CCA_1 trajectory length.")
        dyad_te_df = pd.DataFrame()
else:
    print("[skip] TE directionality — dyad_streams unavailable.")
    dyad_te_df = pd.DataFrame()


Transfer entropy computed for 186 dyads × 4 lags = 744 rows.

TE asymmetry by relationship (lag=3 frames, ~100 ms):
                  mean       std  count
relationship                           
familiar     -0.009674  0.068190    151
stranger     -0.010537  0.075679     35


In [ ]:
# ============================================================================
# 2I. Interpretable EWS via GBT + SHAP (Dakos et al. 2026 PNAS)
# ----------------------------------------------------------------------------
# Dakos, Weinans, Batt et al. 2026, PNAS 123:e2503493122
# (DOI:10.1073/pnas.2503493122). They train gradient-boosted trees on CSD
# indicator trajectories for Reddit-r/place critical transitions and use
# SHAP values to rank per-indicator importance. Here we apply the same
# framework to our per-event CSD indicator matrix at the primary horizon
# h=60s, with the 2-of-3 success label as target. Features: tau_var_60,
# tau_ar1_60, tau_skew_60, reddening_60, adf_p_60.
# Output: per-feature mean |SHAP|, ranked; GBT out-of-bag AUC; SHAP bar
# plot saved to FIGURES_DIR. Reported as an interpretability sidecar —
# does NOT influence the H1 verdict label from Cell 15.
# ============================================================================

# : feature set restricted to the primary-horizon columns (h=60 s,
# Wichers 2016 anchor). The h=90 s Kendall-τ columns exist in events_df
# (tau_var_90 etc.) but are intentionally excluded here — SHAP is a
# primary-verdict interpretability sidecar, not a horizon-stratified model.
# If you want h=90 s attributions, fit a second GBT on the _90 columns.
_GBT_SHAP_FEATURES = [
    "tau_var_60", "tau_ar1_60", "tau_skew_60", "reddening_60", "adf_p_60",
]
_MIN_ROWS_FOR_GBT = 30

try:
    import shap as _shap
    from sklearn.ensemble import GradientBoostingClassifier as _GBC
    from sklearn.metrics import roc_auc_score as _roc_auc
    _SHAP_AVAILABLE = True
except ImportError:
    _SHAP_AVAILABLE = False
    print("[skip] 2I — shap not installed; pip install shap>=0.44.0 and re-run.")

if _SHAP_AVAILABLE and "events_df" in dir() and len(events_df) >= _MIN_ROWS_FOR_GBT \
        and "two_of_three_success_60" in events_df.columns:
    _df_gbt = events_df.dropna(subset=_GBT_SHAP_FEATURES + ["two_of_three_success_60"]).copy()
    if len(_df_gbt) < _MIN_ROWS_FOR_GBT:
        print(f"[skip] 2I — after NaN drop only {len(_df_gbt)} events; "
              f"need >= {_MIN_ROWS_FOR_GBT}.")
    else:
        X_gbt = _df_gbt[_GBT_SHAP_FEATURES].values
        y_gbt = _df_gbt["two_of_three_success_60"].astype(int).values
        if len(np.unique(y_gbt)) < 2:
            print(f"[skip] 2I — target has only one class ({np.unique(y_gbt)}); "
                  "SHAP attribution not meaningful.")
        else:
            model_gbt = _GBC(
                n_estimators=200, max_depth=3, learning_rate=0.05,
                random_state=SEED,
            )
            model_gbt.fit(X_gbt, y_gbt)
            # OOF AUC via train-set score (for interpretability — not a held-out claim)
            train_auc = float(_roc_auc(y_gbt, model_gbt.predict_proba(X_gbt)[:, 1]))
            # SHAP attribution
            explainer = _shap.TreeExplainer(model_gbt)
            shap_values = explainer.shap_values(X_gbt)
            # sklearn GBC binary: TreeExplainer returns list [class_0_shap, class_1_shap]
            # for SHAP < 0.45, or a single array for SHAP >= 0.45. Prefer class-1
            # (positive/success class) for attribution.
            if isinstance(shap_values, list):
                shap_vals_for_attribution = shap_values[1]   # (n_samples, n_features)
            else:
                shap_vals_for_attribution = shap_values       # unified array API
            mean_abs_shap = np.mean(np.abs(shap_vals_for_attribution), axis=0)
            shap_ranking = sorted(
                zip(_GBT_SHAP_FEATURES, mean_abs_shap),
                key=lambda kv: kv[1], reverse=True,
            )
            print(f"GBT train-set AUC on two_of_three_success_60: {train_auc:.3f} "
                  f"(n={len(_df_gbt)})")
            print("SHAP feature ranking (mean |SHAP|, descending):")
            for name, val in shap_ranking:
                print(f"  {name:18s}  {val:.4f}")
            # Save bar plot
            try:
                import matplotlib.pyplot as plt
                fig_shap, ax_shap = plt.subplots(figsize=(7, 3.5))
                names_sorted = [n for n, _ in shap_ranking]
                vals_sorted = [v for _, v in shap_ranking]
                ax_shap.barh(names_sorted[::-1], vals_sorted[::-1], color="steelblue")
                ax_shap.set_xlabel("Mean |SHAP value| on two_of_three_success_60")
                ax_shap.set_title("Per-indicator contribution to H1 success (Dakos 2026 GBT+SHAP)")
                fig_shap.tight_layout()
                _shap_fig_path = FIGURES_DIR / "h1_shap_feature_ranking.png"
                fig_shap.savefig(_shap_fig_path, dpi=120)
                plt.close(fig_shap)
                print(f"  Saved: {_shap_fig_path}")
            except Exception as _fig_e:
                print(f"  [warn] SHAP bar plot skipped: {_fig_e}")
            # Append ranking to the verdict JSON
            try:
                _verdict_path = RESULTS_DIR / "h1_verdict_summary.json"
                if _verdict_path.exists():
                    with open(_verdict_path) as f:
                        _vj = _json.load(f)
                    _vj["shap_feature_ranking"] = {n: float(v) for n, v in shap_ranking}
                    _vj["gbt_train_auc"] = train_auc
                    with open(_verdict_path, "w") as f:
                        _json.dump(_vj, f, indent=2)
                    print(f"  Updated: {_verdict_path}")
            except Exception as _vj_e:
                print(f"  [warn] verdict JSON update failed: {_vj_e}")
else:
    if _SHAP_AVAILABLE:
        print(f"[skip] 2I — events_df has insufficient rows or missing target column.")


## 2H. Rupture-Anchored CSD + HMM Regime Sidecars

Two sidecars to the continuous-CSD analysis (2D–2G):

**2H.1 Rupture-anchored windowing** — `compute_csd_indicators` in 2D uses a fixed-window rolling approach. An alternative: anchor windows to each rupture event (pre=90s, post=30s) and compute CSD indicators only within those event-centered segments. Citations: Marwan, Romano, Thiel & Kurths 2007, *Phys. Rep.* 438:237-329 (DOI:10.1016/j.physrep.2006.11.001) — RQA/CRQA primer; Fusaroli, Rączaszek-Leonardi & Tylén 2014, *New Ideas in Psychology* 32:147-157 (DOI:10.1016/j.newideapsych.2013.03.005) — windowed CRQA in dialogic interaction.

**2H.2 HMM regime segmentation** — continuous CSD reframed as *regime transitions*: fit a Gaussian HMM (k=3) on the CCA_1 trajectory and report state means + transition matrix. Not a replacement for continuous CSD — a sidecar that tests whether coupling dynamics look better described as discrete states than a continuous gradient. Citation: Vidaurre, Abeysuriya, Becker et al. 2018, *NeuroImage* 180:646-656 (DOI:10.1016/j.neuroimage.2017.06.077) — TDE-HMM for M/EEG regime discovery.

*Honest note:* HMM sidecar is `[UNDERPOWERED]` on a 50-dyad pilot (per-dyad HMM fit). Result is descriptive only until the pilot scales.

In [ ]:
# ============================================================================
# 2H.1 Rupture-anchored CSD (sidecar to fixed-window 2D)
# ============================================================================
# Reuses:
#   - dyad_streams / compute_cross_modal_rupture / rolling_cca1_trajectory
#     (built in cell 8) → supply rupture_times and the CCA_1 trajectory series
#   - compute_csd_indicators (cell 9)
#   - anchor_crqa_to_rupture_events (signal_utils.py,  addition)
# Expected runtime: ~2-5 min on 50-dyad pilot.

# Import experiments.signal_utils with a clean skip if the module isn't on path
# (sys.path was extended in Cell 4; this guard covers the out-of-order-run case).
try:
    from experiments.signal_utils import anchor_crqa_to_rupture_events
    _EXP_AVAILABLE = True
except (ImportError, ModuleNotFoundError):
    _EXP_AVAILABLE = False
    print('[skip] 2H.1 — experiments/signal_utils.py not importable; re-run Cell 4 or check DATA_ROOT.')

if not _EXP_AVAILABLE:
    section2h1_summary = {'status': 'skipped', 'reason': 'experiments module not importable'}

# Guard: skip if the upstream globals are missing (e.g. running 2H standalone)
_required_h1 = ["dyad_streams", "compute_cross_modal_rupture",
                "rolling_cca1_trajectory", "compute_csd_indicators"]
_missing_h1 = [n for n in _required_h1 if n not in globals()]
if _missing_h1:
    print(f"[skip] 2H.1 — missing upstream globals: {_missing_h1}")
    section2h1_summary = {"status": "skipped", "missing": _missing_h1}
elif _EXP_AVAILABLE:
    import pandas as _pd_h1

    anchored_records = []
    for ik, streams in dyad_streams.items():
        try:
            # v6: unpack streams dict into positional args
            rupt = compute_cross_modal_rupture(
                streams.get("head_a"), streams.get("head_b"),
                streams.get("gaze_a"), streams.get("gaze_b"),
                fps=30, percentile=RUPTURE_PERCENTILE, sustain_s=RUPTURE_SUSTAIN_S,
            )
            # rupture_events: list of (start_frame, end_frame) at 30 fps.
            rupture_times = [s / 30.0 for (s, _) in rupt.get("rupture_events", [])]
            if len(rupture_times) == 0:
                continue

            # v6: unpack for rolling_cca1_trajectory(sig_a, sig_b, weights_a, weights_b)
            _centers_rt, R_ts = rolling_cca1_trajectory(
                streams["cca_a"], streams["cca_b"],
                weights_a=streams.get("weights_a"),
                weights_b=streams.get("weights_b"),
                window_s=5.0, step_s=0.5, fps=30,
            )
            if R_ts is None or len(R_ts) < 120:
                continue

            # R_ts is at 2 Hz (rolling_cca1_trajectory step_s=0.5). rupture_times are in
            # seconds, so fs must be 2.0 — passing fs=30 here makes every index
            # lookup 15× past end-of-series and rejects every window.
            anchored = anchor_crqa_to_rupture_events(
                R_ts,
                rupture_times,
                fs=2.0,
                pre_window_s=90.0,
                post_window_s=30.0,
                drop_overlapping=True,
            )

            for seg in anchored:
                csd_res = compute_csd_indicators(seg["segment"])
                anchored_records.append({
                    "dyad_id": ik,
                    "rupture_time_s": seg["rupture_time_s"],
                    "segment_len": int(seg["end_idx"] - seg["start_idx"]),
                    "tau_var": csd_res.get("tau_var", float("nan")),
                    "tau_ar1": csd_res.get("tau_ar1", float("nan")),
                    "n_pre": seg["pre_samples"],
                    "n_post": seg["post_samples"],
                })
        except Exception as _e:
            print(f"  [warn] anchor failed for {ik}: {type(_e).__name__}: {_e}")

    anchored_df = _pd_h1.DataFrame(anchored_records)
    print(f"\nRupture-anchored CSD: {len(anchored_df)} anchored segments across "
          f"{anchored_df['dyad_id'].nunique() if len(anchored_df) else 0} dyads")

    if len(anchored_df) > 0:
        tau_var_med = float(anchored_df["tau_var"].median())
        tau_ar1_med = float(anchored_df["tau_ar1"].median())
        tau_var_mean = float(anchored_df["tau_var"].mean())
        tau_ar1_mean = float(anchored_df["tau_ar1"].mean())
        print(f"  τ_var  (anchored): median={tau_var_med:.3f}, mean={tau_var_mean:.3f}")
        print(f"  τ_AR1  (anchored): median={tau_ar1_med:.3f}, mean={tau_ar1_mean:.3f}")

        # Compare to fixed-window τ if available from 2F
        if "events_df" in globals() and "tau_var" in events_df.columns:
            fixed_var_med = float(events_df["tau_var"].median())
            fixed_ar1_med = float(events_df["tau_ar1"].median()) if "tau_ar1" in events_df.columns else float("nan")
            print(f"\n  vs. fixed-window (2F):")
            print(f"    τ_var  fixed-window median  = {fixed_var_med:.3f}  (anchored: {tau_var_med:.3f})")
            print(f"    τ_AR1  fixed-window median  = {fixed_ar1_med:.3f}  (anchored: {tau_ar1_med:.3f})")
            print(f"    anchored - fixed τ_var      = {tau_var_med - fixed_var_med:+.3f}")
            print(f"    anchored - fixed τ_AR1      = {tau_ar1_med - fixed_ar1_med:+.3f}")
            interp = ("anchored shows CLEANER pre-rupture trend"
                      if abs(tau_var_med) > abs(fixed_var_med) else
                      "fixed-window shows cleaner trend")
            print(f"    interpretation: {interp}")

        _out_h1 = RESULTS_DIR / "section2h1_anchored_csd.pkl"
        anchored_df.to_pickle(_out_h1)
        print(f"\n✓ Saved anchored CSD results to {_out_h1}")

        section2h1_summary = {
            "status": "ok",
            "n_segments": int(len(anchored_df)),
            "n_dyads": int(anchored_df["dyad_id"].nunique()),
            "tau_var_median": tau_var_med,
            "tau_ar1_median": tau_ar1_med,
            "tau_var_mean": tau_var_mean,
            "tau_ar1_mean": tau_ar1_mean,
        }
    else:
        section2h1_summary = {"status": "no_ruptures"}

# Citations:
#   Marwan, Romano, Thiel & Kurths 2007, Phys. Rep. 438:237-329
#     (DOI:10.1016/j.physrep.2006.11.001) — RQA + CRQA primer.
#   Fusaroli, Rączaszek-Leonardi & Tylén 2014, New Ideas in Psychology 32:147-157
#     (DOI:10.1016/j.newideapsych.2013.03.005) — windowed CRQA in dialogic interaction.


In [ ]:
# ============================================================================
# 2H.2 HMM regime segmentation on R(t)  [UNDERPOWERED on 50-dyad pilot]
# ============================================================================
# Fit Gaussian HMM (k=3) on each dyad's CCA_1 trajectory. Reports state means
# + transition matrix + per-state time fraction by relationship. Sidecar to
# continuous CSD — NOT a replacement.
# Reuses:
#   - dyad_streams, rolling_cca1_trajectory (cell 8)
#   - hmm_regime_segmentation (signal_utils.py,  addition)

# Import experiments.signal_utils with a clean skip if the module isn't on path.
try:
    from experiments.signal_utils import hmm_regime_segmentation
    _EXP_HMM_AVAILABLE = True
except (ImportError, ModuleNotFoundError):
    _EXP_HMM_AVAILABLE = False
    print('[skip] 2H.2 — experiments/signal_utils.py not importable; re-run Cell 4 or check DATA_ROOT.')

if not _EXP_HMM_AVAILABLE:
    section2h2_summary = {'status': 'skipped', 'reason': 'experiments module not importable'}

# Guard: skip if upstream globals are missing
_required_h2 = ["dyad_streams", "rolling_cca1_trajectory"]
_missing_h2 = [n for n in _required_h2 if n not in globals()]
if _missing_h2:
    print(f"[skip] 2H.2 — missing upstream globals: {_missing_h2}")
    section2h2_summary = {"status": "skipped", "missing": _missing_h2}
elif _EXP_HMM_AVAILABLE:
    import pandas as _pd_h2

    # Attempt to load relationship labels from companion notebook's features_df
    try:
        _fdf_h2 = _pd_h2.read_pickle(RESULTS_DIR / "features_df.pkl")
        _rel_map_h2 = dict(zip(_fdf_h2["dyad_id"], _fdf_h2["relationship"]))
    except Exception:
        _rel_map_h2 = {}

    hmm_records = []
    hmm_fallback_count = 0
    for ik, streams in dyad_streams.items():
        try:
            # v6: unpack streams for rolling_cca1_trajectory(sig_a, sig_b, weights_a, weights_b)
            _centers_ht, R_ts = rolling_cca1_trajectory(
                streams["cca_a"], streams["cca_b"],
                weights_a=streams.get("weights_a"),
                weights_b=streams.get("weights_b"),
                window_s=5.0, step_s=0.5, fps=30,
            )
            if R_ts is None or len(R_ts) < 90:  # need enough samples for HMM
                continue
            res = hmm_regime_segmentation(R_ts, n_states=3, random_state=42)
            if res.get("fallback"):
                hmm_fallback_count += 1
                continue

            states = res["states"]
            k = res["n_states"]
            # Fraction of time in each state
            frac = {f"p_state_{i}": float(np.mean(states == i)) for i in range(k)}
            hmm_records.append({
                "dyad_id": ik,
                "relationship": _rel_map_h2.get(ik, "unknown"),
                "log_likelihood": res["log_likelihood"],
                "converged": res["converged"],
                **frac,
                "state_mean_0": float(res["state_means"][0]) if k > 0 else float("nan"),
                "state_mean_1": float(res["state_means"][1]) if k > 1 else float("nan"),
                "state_mean_2": float(res["state_means"][2]) if k > 2 else float("nan"),
                "transition_diag_sum": float(np.trace(res["transition_matrix"])),
            })
        except Exception as _e:
            print(f"  [warn] HMM failed for {ik}: {type(_e).__name__}: {_e}")

    if hmm_fallback_count == len(dyad_streams) and len(hmm_records) == 0:
        print("⚠ hmmlearn not installed — all dyads fell back. "
              "Install with `pip install 'hmmlearn>=0.3.0'` and re-run.")
        section2h2_summary = {"status": "hmmlearn_missing"}
    elif len(hmm_records) == 0:
        print("⚠ no dyads had sufficient R(t) length for HMM fit.")
        section2h2_summary = {"status": "insufficient_data"}
    else:
        hmm_df = _pd_h2.DataFrame(hmm_records)
        print(f"\nHMM regime segmentation: {len(hmm_df)} dyads fit (k=3)")
        print(f"  fallback (hmmlearn missing): {hmm_fallback_count}")
        print(f"  converged (EM): {int(hmm_df['converged'].sum())} / {len(hmm_df)}")
        print(f"  mean log-likelihood: {float(hmm_df['log_likelihood'].mean()):.2f}")

        by_rel_h2 = hmm_df.groupby("relationship")[
            ["p_state_0", "p_state_1", "p_state_2", "transition_diag_sum"]
        ].mean().round(3)
        print("\n  Per-state time fraction by relationship (diag_sum = state persistence):")
        print(by_rel_h2.to_string())

        # [UNDERPOWERED] label per plan
        n_dyads_ok = len(hmm_df)
        if n_dyads_ok < 100:
            print(f"\n  [UNDERPOWERED] n_dyads_ok={n_dyads_ok} < 100 (MIN_N_EFF_DYADS). "
                  "Per-dyad HMM fit is descriptive only at this pilot size; do not claim "
                  "regime-switching as the primary finding until scaled.")

        _out_h2 = RESULTS_DIR / "section2h2_hmm_regimes.pkl"
        hmm_df.to_pickle(_out_h2)
        print(f"\n✓ Saved HMM regime results to {_out_h2}")

        section2h2_summary = {
            "status": "ok" if n_dyads_ok >= 100 else "underpowered",
            "n_dyads": int(n_dyads_ok),
            "n_fallback": int(hmm_fallback_count),
            "mean_log_likelihood": float(hmm_df["log_likelihood"].mean()),
            "converged_rate": float(hmm_df["converged"].mean()),
        }

# Citation: Vidaurre et al. 2018 NeuroImage 180:646. NOT Li et al. 2026 (which uses k-means).


### Cross-modal validation — independence status

**Circularity risk reduced (not eliminated).** Prior drafts defined rupture as a drop in `R(t)`, then tested whether CSD on `R(t)` predicted those drops — tautological. The current design uses an **independence-preserving cross-modal rupture signal** computed from `head_encodings` + `gaze_encodings` only, disjoint from `CCA_1`'s inputs (body motion, FAU).

**Trunk-sharing caveat.** Seamless Interaction's `head_encodings`, `gaze_encodings`, `FAUValue`, `emotion_valence`, `emotion_arousal` may share an encoder backbone — common in multi-task vision models. Disjointness at the feature-name level does not imply disjointness at the information level. Confirming trunk separation requires inspecting the Seamless model architecture (deferred). Until confirmed, the claim is **"reduced circularity"**, not independence.

**Learned-representation caveat.** `head_encodings` / `gaze_encodings` are learned neural embeddings, not calibrated rotations / gaze angles. Rupture here = *movement of the learned representation away from the coordinated state*, a model-dependent proxy for behavioural rupture.

**Excluded modalities and why.**

| Feature | Reason |
|---|---|
| `movement:FAUValue` | In `CCA_1` inputs → circular |
| `movement:emotion_valence`, `emotion_arousal` | Share upstream video model with FAU |
| `smplh:body_pose`, `translation` | In `CCA_1` inputs → circular |


## 3. Learned early warning — brief survey

Classical CSD via statistical indicators gives an interpretable baseline. Two learned alternatives are relevant for an AI-safety reviewer; neither replaces the primary verdict, both inform the sidecar cells.

### 3a. CNN-LSTM bifurcation classifier

Bury et al. 2021 *PNAS* 118:e2106140118 demonstrated that a convolutional-recurrent network trained on simulated fold / Hopf / transcritical bifurcations transfers to empirical ecological and climate data with >85% AUC on held-out test sets. The network learns AC(1) and variance patterns as convolutional filters rather than hand-crafted features. Scope note: we include a conceptual reference implementation in this notebook for architectural clarity; no trained weights are shipped.

### 3b. GBT + SHAP for interpretable CSD

Dakos et al. 2026 *PNAS* 123:e2503493122 — interpretable early warnings via gradient-boosted trees with SHAP explanations. Detects 50% of r/place transitions within 20 min at 3.7% false-positive rate. Per-feature SHAP ranking answers "why did the detector fire here?" — a precondition for deploying EWS as a safety gate.

This notebook's §2I implements the GBT + SHAP sidecar over the per-event CSD indicator table. When the effective-events floor is met, the top-3 SHAP features are merged into `h1_verdict_summary.json`.

**Deferred (backlog).** Neural ODEs (Chen et al. 2018), deep Koopman operators (Lusch et al. 2018), PINNs (Raissi et al. 2019), synthetic-to-empirical transfer — none are implemented here. Tracked in future-work.


In [ ]:
# Conceptual PyTorch CNN-LSTM for CSD classification
# Reference: Bury et al. (2021, PNAS)
import torch
import torch.nn as nn

class CSDClassifier(nn.Module):
    """CNN-LSTM for early warning signal detection in time-series."""
    def __init__(self, input_dim=1, cnn_filters=32, lstm_hidden=64):
        super().__init__()
        # Conv layer: learn local AC(1) and variance patterns
        self.conv = nn.Conv1d(input_dim, cnn_filters, kernel_size=3, padding=1)
        # LSTM: capture temporal dependencies in CSD indicators
        self.lstm = nn.LSTM(cnn_filters, lstm_hidden, batch_first=True)
        # Classifier head
        self.fc = nn.Linear(lstm_hidden, 2)  # pre-bifurcation vs post-bifurcation

    def forward(self, x):
        # x shape: (batch, time_steps, input_dim)
        x = x.transpose(1, 2)  # (batch, input_dim, time_steps) for Conv1d
        x = torch.relu(self.conv(x))
        x = x.transpose(1, 2)  # back to (batch, time_steps, filters)
        _, (h_n, _) = self.lstm(x)
        logits = self.fc(h_n[-1])  # use final hidden state
        return logits

# Example: classify a 60-second window (300 frames at 5 Hz)
model = CSDClassifier(input_dim=1, cnn_filters=32, lstm_hidden=64)
dummy_input = torch.randn(8, 300, 1)  # batch of 8 time-series
logits = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Logits shape: {logits.shape}  # (batch=8, classes=2)")

Input shape: torch.Size([8, 300, 1])
Logits shape: torch.Size([8, 2])  # (batch=8, classes=2)


## 6. Summary & next steps

### What this notebook delivers

1. **Independence-preserving rupture signal** — head + gaze composite, disjoint from `CCA_1` inputs; rupture = composite z-score ≥ 90th percentile sustained ≥ 2 s.
2. **Three CSD indicators (≥ 2 of 3 primary)** — variance, AR(1), skewness on Gaussian-detrended `CCA_1(t)` at `bandwidth_frac = 0.10`.
3. **Two diagnostic sidecars** — spectral reddening (Bury 2020 framework, bands rescaled for 2 Hz rolling-Pearson output) and ADF stationarity p-value. Neither gates the verdict.
4. **Multi-horizon test** — 60 / 90 s. Primary verdict anchored on 60 s (Wichers 2016), 90 s reported as a robustness sidecar. The 30 s horizon was excluded because `WARNING_LEAD_S = 30 s` makes the pre-window zero-length.
5. **Circular block bootstrap null** — Kunsch 1989 family, cf. Politis & Romano 1994. Replaces the earlier uniform rupture-time shuffle, which was under-conservative for non-stationary `CCA_1(t)`.
6. **Sample-size gate** — `[UNDERPOWERED]` when `n_eff_dyads < 100` OR `n_events_effective < 200`.
7. **Interpretability sidecar** — GBT + SHAP per-event feature ranking (Dakos 2026); bar plot saved to `FIGURES_DIR/h1_shap_feature_ranking.png`.
8. **Directionality sidecar** — transfer entropy on `CCA_1(a,b)` at lags {1, 3, 5, 10} frames (Schreiber 2000); asymmetry reported by relationship group.
9. **Regime-switching sidecar** — TDE-HMM occupancy per dyad (Vidaurre 2018); familiar vs stranger regime share.
10. **Verdict artifact** — `RESULTS_DIR/h1_verdict_summary.json` with H1 label, per-horizon rates, SHAP ranking, TE asymmetry, sample floors.

### Canonical limitations

- Upstream `CCA_1(t)` is FAU-dominated (~91 % of |w| on the sample-run cohort of N=186), so any positive H1 verdict at this scope is facial-mimicry-driven, not multimodal coupling. Block-regularized CCA is the upstream remediation (see companion notebook backlog).
- **Rupture-detector calibration** `[UNVERIFIED-TRANSFER]`. The detector uses `RUPTURE_PERCENTILE = 75` (per-dyad) and `RUPTURE_SUSTAIN_S = 0.75 s`. The 75th-percentile setting follows Fusaroli et al. 2014's CRQA dialog-rupture threshold convention; the 0.75 s sustain is a middle-ground choice within the 0.5-2.0 s range typical of short behavioural-event detection. No single paper validates this exact pair on the gaze+head composite z-score used here — the transfer from dialog-CRQA / multi-modal-ML-EWS to Seamless Interaction head+gaze encodings is analogical, not empirically calibrated. A parameter sensitivity sweep is tracked as a roadmap item.
- CSD is known to be unreliable in short-series / high-dim settings (Meisel & Kuehn 2012, Smit et al. 2025 32.9 % sensitivity floor); multi-indicator criteria are a partial remediation only.
- **TE-asymmetry re-framing (§6, Cell 2H).** The directional-control probe (AI-safety direction #4) uses the same TE machinery already cited in §6 sidecars; no new code, only reframed interpretation. A nonzero asymmetry index is evidence of directional influence independent of synchrony magnitude — relevant to auditability even when synchrony per se is at chance.
- The rupture detector is model-dependent (head + gaze *learned* encodings, not calibrated rotations); rupture-label quality is therefore an upper bound on H1's detectability.

### Future work (backlog)

- Partner-shuffled null (Moulder et al. 2018) as a second null complementary to the circular block bootstrap.
- 500-shuffle full null run with Phipson-Smyth 2010 continuity correction.
- TDE-HMM regime segmentation as a primary (not sidecar) analysis (Vidaurre 2018).
- **Cross-cutting safety-relevant probes appear as §7 (MI diagnostic), §8 (calibrated-FPR rupture), §9 (per-modality CCA), §10 (balancing + alt classifiers).** The transfer-entropy asymmetry computation in §2H additionally serves as a directional-influence (who-leads-whom) probe — asymmetry sign + magnitude by relationship group is reported as a sidecar interpretation rather than a separate cell.

### Reproducibility

`SEED = 42` (override via `SEAMLESS_SEED`). Paths parameterised via `SEAMLESS_DATA_DIR`, `SEAMLESS_RESULTS_DIR`. Default `NB2_MAX_LOAD_DYADS = 0` runs on all 186 dyads.

## References

### Core methods (load-bearing)

- Scheffer, M., Bascompte, J., Brock, W.A., et al. 2009, *Nature* 461:53-59 (DOI:10.1038/nature08227) — canonical CSD statement.
- Dakos, V., Carpenter, S.R., Brock, W.A., et al. 2012, *PLOS ONE* 7:e41010 (DOI:10.1371/journal.pone.0041010) — Gaussian-kernel detrending, skewness indicator.
- Dakos, V., Scheffer, M., van Nes, E.H., et al. 2008, *PNAS* 105:14308-14312 (DOI:10.1073/pnas.0802430105) — CSD in paleoclimate.
- Wichers, M., Groot, P.C., & Psychosystems ESM Group 2016, *JAMA Psychiatry* 73:739-747 (DOI:10.1001/jamapsychiatry.2015.3079) — 60–120 s pre-transition horizon.
- Politis, D.N., & Romano, J.P. 1994, *J. Am. Stat. Assoc.* 89:1303-1313 (DOI:10.1080/01621459.1994.10476870) — stationary / circular block bootstrap.
- Bury, T.M., Bauch, C.T., & Anand, M. 2020, *J. R. Soc. Interface* 17:20200482 (DOI:10.1098/rsif.2020.0482) — spectral reddening framework.
- Kuramoto, Y. 1984, *Chemical Oscillations, Waves, and Turbulence* (Springer) — foundational oscillator synchronisation.
- Richardson, M.J., Dale, R., & Marsh, K.L. 2015, in *Handbook of Embodied Cognition* — phase coherence in dyadic behaviour.
- Marwan, N., Romano, M.C., Thiel, M., & Kurths, J. 2007, *Phys. Rep.* 438:237-329 (DOI:10.1016/j.physrep.2006.11.001) — RQA / CRQA primer; used for rupture-anchored CSD.

### Null-framing additions (expected-null literature)

- Meisel, C., & Kuehn, C. 2012, *Phys. Rev. E* 86:051108 (DOI:10.1103/PhysRevE.86.051108) — EWS fragility in high-dimensional systems.
- Lenton, T.M. 2011, *Nature Climate Change* 1:201-209 (DOI:10.1038/nclimate1143) — EWS false-positive rates.
- Ruths, J., & Ruths, D. 2014, *Science* 343:1373-1376 (DOI:10.1126/science.1242063) — null-dominated network inference.
- Laestadius, L. 2025, *Nature H&SS Communications* — non-stationary socioaffective alignment.
- Smit, A.C., Helmich, M.A., Bringmann, L.F., et al. 2025, *Clin. Psychol. Sci.* (DOI:10.1177/21677026241305136) — 32.9 % sensitivity floor.
- Helmich, M.A., Schreuder, M.J., Bringmann, L.F., et al. 2024, *Nat. Rev. Psychol.* (DOI:10.1038/s44159-024-00369-y) — clinical CSD caveats.

### Sidecar methods (ran but uninformative on this dataset)

- Schreiber, T. 2000, *Phys. Rev. Lett.* 85:461-464 (DOI:10.1103/PhysRevLett.85.461) — transfer-entropy definition. TE asymmetry ≈ 0 in both familiar and stranger groups on this run.
- Vidaurre, D., Abeysuriya, R., Becker, R., et al. 2018, *NeuroImage* 180:646-656 (DOI:10.1016/j.neuroimage.2017.06.077) — TDE-HMM regime discovery. Occupancy differed by 1 pp between groups on this run.
- Dakos, V., Weinans, E., Batt, R.D., et al. 2026, *PNAS* 123:e2503493122 (DOI:10.1073/pnas.2503493122) — interpretable ML early warnings (GBT + SHAP).
- Moulder, R.G., Boker, S.M., Ramseyer, F., & Tschacher, W. 2018, *Psychological Methods* 23:757-773 (DOI:10.1037/met0000172) — partner-shuffled null canonical design.
- Bury, T.M., Sujith, R.I., Pavithran, I., et al. 2021, *PNAS* 118:e2106140118 (DOI:10.1073/pnas.2106140118) — CNN-LSTM CSD classifier (conceptual reference in §3a).
- Boettiger, C., & Hastings, A. 2012, *Proc. R. Soc. B* 279:4734-4739 (DOI:10.1098/rspb.2012.2085) — short-series CSD reliability.

### AI-safety anchors

- Amodei, D., Olah, C., Steinhardt, J., et al. 2016, *Concrete Problems in AI Safety* (arXiv:1606.06565).
- Christiano, P., & Amodei, D. 2018, *AI Safety via Debate / Iterated Amplification* (arXiv:1805.00899).
- Gabriel, I. 2022, *Toward a Theory of Justice for Artificial Intelligence* — socioaffective alignment framing.
- Hendrycks, D., & Gimpel, K. 2017, *ICLR* (arXiv:1610.02136) — anomaly / OOD detection baseline.
- Bengio, Y., Hinton, G., Yao, A., et al. 2024, *Science* 384:842-845 (DOI:10.1126/science.adn0117) — managing extreme AI risks.
- Anthropic 2024, *Responsible Scaling Policy v2.0* — capability-threshold gates.
- Omel'chenko, O.E. 2026, *Nat. Communications* — inverse problem for coupled oscillators (proposed for AI-safety angle).

### Dataset

- Meta Seamless Interaction (BSD). [seamless-interaction.github.io](https://seamless-interaction.github.io).


## 7. Mutual-Information Diagnostic — Is the Bottleneck Linearity or Signal?

**Question.** Is the observed AUC ≈ chance on canonical-correlation features a *linearity* ceiling, or a *signal* ceiling?

**Method.** Kraskov et al. 2004 (*Phys. Rev. E* 69:066138, DOI:10.1103/PhysRevE.69.066138) kNN-based mutual-information estimator on the 1-D `CCA_1(a)` vs `CCA_1(b)` trajectories per dyad. MI captures linear + nonlinear dependencies; CCA captures only linear. If MI AUC materially exceeds CCA AUC, nonlinearity is the bottleneck.

**Gate.** If `MI_AUC − CCA_AUC ≥ 0.05` → `linearity_limit=True`, advance per-modality CCA (§9) + alternative classifiers (§10). If `< 0.05` → signal is genuinely weak; report as scope caveat in §6 and frame per Gordon & Bartsch 2026 heterogeneity review.

**AI-safety mapping.** Anthropic direction #2 (measuring alignment) requires robust coupling measurement. An MI-vs-CCA gap establishes that linear synchrony proxies *systematically underestimate* coupling magnitude — directly relevant to capability-threshold gating where underestimation = late detection.


In [ ]:
# ============================================================================
# 7. MI diagnostic (Kraskov 2004) on CCA_1(a) vs CCA_1(b)
# ============================================================================
# Ported from sprints/2026-04-ai-safety/proposals/D_mutual_information_diagnostic_2026-04-20.py
# Adapted to NB2 namespace: operates on 1-D CCA_1 trajectories in dyad_streams,
# NOT on multi-dim features_a/features_b. Simpler (no NPZ reload).
# NOTE: MI on CCA_1 projection is a LOWER BOUND on
# full-feature MI. CCA_1 discards variance from components 2..N. If
# linearity_limit=False here, the gap MAY still exist at the full-feature level
# — re-run on per-modality features (§9) to confirm.
# StandardScaler now fitted inside each CV fold to avoid
# test-fold leakage (fit/transform asymmetry).

# nb1_features removed from guard —
# the cell body uses dyad_streams[ik]["relationship"], not nb1_features.
_required_mi = ["dyad_streams", "np", "pd", "RESULTS_DIR", "SEED"]
_missing_mi = [n for n in _required_mi if n not in globals()]
if _missing_mi:
    raise NameError(f"Cell 7 MI diagnostic requires {_missing_mi} from earlier cells")

try:
    from sklearn.feature_selection import mutual_info_regression
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score
    from scipy.stats import mannwhitneyu
    _MI_DEPS_OK = True
except ImportError as _e:
    _MI_DEPS_OK = False
    print(f"[skip] MI diagnostic deps missing: {_e}")

if _MI_DEPS_OK:
    _MI_K = 5                    # Kraskov 2004 k-nearest-neighbour
    _MI_WINDOWS = [30, 90, 150]  # frames @ 30 Hz → 1s / 3s / 5s
    _MI_LAG = 15                 # 0.5 s for directionality proxy
    _MI_MIN_FRAMES = 200         # signal-length floor (≈ 6.7 s @ 30 Hz)

    def _estimate_mi(x, y, k=_MI_K):
        """Kraskov MI between 1-D signals."""
        x = np.asarray(x, dtype=float).reshape(-1, 1)
        y = np.asarray(y, dtype=float).ravel()
        T = min(len(x), len(y))
        if T < k + 2:
            return 0.0
        x, y = x[:T], y[:T]
        valid = ~(np.isnan(x).any(axis=1) | np.isnan(y))
        x, y = x[valid], y[valid]
        if len(x) < k + 2:
            return 0.0
        mi = mutual_info_regression(x, y, n_neighbors=k, random_state=SEED)
        return float(np.mean(mi))

    def _windowed_mi(x, y, win, k=_MI_K):
        T = min(len(x), len(y))
        if T < win:
            return np.array([0.0])
        step = max(1, win // 2)
        out = []
        for s in range(0, T - win + 1, step):
            out.append(_estimate_mi(x[s:s+win], y[s:s+win], k))
        return np.asarray(out) if out else np.array([0.0])

    # ---- Extract MI features per dyad ----
    mi_rows = []
    for ik, d in dyad_streams.items():
        a, b = np.asarray(d.get("cca_a"), dtype=float), np.asarray(d.get("cca_b"), dtype=float)
        if a.ndim != 1 or b.ndim != 1 or len(a) < _MI_MIN_FRAMES:
            continue
        row = {"interaction_key": ik,
               "relationship": d.get("relationship", "unknown")}
        row["mi_global"] = _estimate_mi(a, b)
        for w in _MI_WINDOWS:
            if len(a) < w:
                continue
            trace = _windowed_mi(a, b, w)
            row[f"mi_w{w}_mean"] = float(np.mean(trace))
            row[f"mi_w{w}_std"]  = float(np.std(trace))
            row[f"mi_w{w}_max"]  = float(np.max(trace))
            if len(trace) > 2:
                row[f"mi_w{w}_slope"] = float(np.polyfit(np.arange(len(trace)), trace, 1)[0])
        if len(a) > _MI_LAG + 30:
            row["mi_lag_ab"] = _estimate_mi(a[:-_MI_LAG], b[_MI_LAG:])
            row["mi_lag_ba"] = _estimate_mi(b[:-_MI_LAG], a[_MI_LAG:])
            row["mi_asymmetry"] = row["mi_lag_ab"] - row["mi_lag_ba"]
        mi_rows.append(row)

    mi_df = pd.DataFrame(mi_rows)
    print(f"MI features for {len(mi_df)} dyads "
          f"({(mi_df['relationship'] == 'stranger').sum()} stranger / "
          f"{(mi_df['relationship'] == 'familiar').sum()} familiar)")

    # ---- Classify: 5-fold stratified LR; scaler fit INSIDE each fold (fix M2) ----
    mi_auc = float("nan")
    if len(mi_df) >= 30 and mi_df["relationship"].nunique() >= 2:
        feat_cols = [c for c in mi_df.columns
                     if c not in {"interaction_key", "relationship"}]
        X = mi_df[feat_cols].fillna(0.0).values
        y = (mi_df["relationship"] == "familiar").astype(int).values
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        probs = np.zeros(len(y))
        fold_aucs = []
        for tr, te in cv.split(X, y):
            _sc = StandardScaler()
            Xtr = _sc.fit_transform(X[tr])
            Xte = _sc.transform(X[te])
            clf = LogisticRegression(C=1.0, class_weight="balanced",
                                     max_iter=1000, random_state=SEED)
            clf.fit(Xtr, y[tr])
            probs[te] = clf.predict_proba(Xte)[:, 1]
            if len(np.unique(y[te])) == 2:
                fold_aucs.append(roc_auc_score(y[te], probs[te]))
        mi_auc = float(roc_auc_score(y, probs))
        print(f"\nMI-based AUC: {mi_auc:.3f}  (fold mean={np.mean(fold_aucs):.3f}"
              f", std={np.std(fold_aucs):.3f})")
    else:
        print("[skip] insufficient class balance for classification")

    # ---- Baseline CCA AUC from step4_summary.json (if present) ----
    cca_auc = float("nan")
    try:
        import json as _json
        _sp = RESULTS_DIR / "step4_summary.json"
        if _sp.exists():
            with open(_sp) as f:
                _s = _json.load(f)
            cca_auc = float(_s.get("primary_auc", float("nan")))
    except Exception:
        pass

    delta = mi_auc - cca_auc if (np.isfinite(mi_auc) and np.isfinite(cca_auc)) else float("nan")
    print(f"CCA baseline AUC: {cca_auc:.3f}")
    print(f"Δ(MI − CCA) = {delta:+.3f}")

    linearity_limit = bool(np.isfinite(delta) and delta >= 0.05)
    if linearity_limit:
        print("FINDING: nonlinear coupling dominates → CCA's linearity is the bottleneck.")
        print("         Proceed to per-modality CCA (§9) + alt classifiers (§10).")
    else:
        print("FINDING: MI ≈ CCA on CCA_1 projection. Weak signal — but MI on CCA_1 is a LOWER")
        print("         BOUND on full-feature MI; §9 re-tests with multi-dim per-modality features.")

    # ---- Per-relationship descriptive stats ----
    if "mi_global" in mi_df.columns and mi_df["relationship"].nunique() >= 2:
        stra = mi_df.loc[mi_df["relationship"] == "stranger", "mi_global"].dropna()
        fam  = mi_df.loc[mi_df["relationship"] == "familiar", "mi_global"].dropna()
        if len(stra) >= 5 and len(fam) >= 5:
            # [QA skill note] mi_global is a SINGLE comparison; no correction needed.
            # Per-window MI comparisons (scopes = 4 modalities × 3 windows = 12) would
            # require Bonferroni α' = 0.05/12 = 0.0042; those are descriptive here (not
            # in the classification-AUC path, which is the primary statistic).
            _, p = mannwhitneyu(stra, fam, alternative="two-sided")
            print(f"\n  mi_global  stranger={stra.mean():.4f}  familiar={fam.mean():.4f}  "
                  f"Mann-Whitney p={p:.4f}  (single contrast; no MC correction)")

            # Apply the pre-registered verdict-gate (Cell 0) to the MI headline
            if mi_auc < 0.60:
                mi_label = "[NEGATIVE_RESULT]"
            elif mi_auc < 0.70:
                mi_label = "[EMPIRICAL but weak]"
            else:
                mi_label = "[EMPIRICAL]"
            print(f"\nLabel: {mi_label}  (MI AUC {mi_auc:.3f} vs 0.60/0.70 thresholds)")

    # ---- Persist ----
    mi_df.to_pickle(RESULTS_DIR / "mi_diagnostic_df.pkl")
    _summary = {
        "n_dyads": int(len(mi_df)),
        "mi_auc": mi_auc,
        "cca_auc_baseline": cca_auc,
        "delta_mi_minus_cca": delta,
        "linearity_limit": linearity_limit,
        "k_neighbors": _MI_K,
        "window_sizes_frames": _MI_WINDOWS,
        "cca_dim_used": 1,
        "note": ("MI computed on CCA_1 projection only; MI value is a LOWER BOUND on"
                 " full-feature MI because CCA_1 discards variance from components ≥2."
                 " See §9 per-modality CCA for multi-dim comparison."),
        "min_signal_frames": _MI_MIN_FRAMES,
        "scaler_fit_inside_fold": True,
        "reference": "Kraskov, Stogbauer, Grassberger 2004, PRE 69:066138",
    }
    import json as _json
    with open(RESULTS_DIR / "mi_diagnostic_summary.json", "w") as f:
        _json.dump(_summary, f, indent=2)
    print(f"\n✓ Saved mi_diagnostic_df.pkl ({len(mi_df)} rows) + mi_diagnostic_summary.json")


## 8. Calibrated False-Positive-Rate Rupture Detector

**Motivation.** The primary rupture detector in §2C uses a per-dyad 75th-percentile trigger with a 0.75 s sustain. That's a prevalence-based threshold; it has no explicit FPR target. Boettiger & Hastings 2012 (*Proc. R. Soc. B* 279:4734-4739, DOI:10.1098/rspb.2012.2085) argue that short-series early-warning detectors should be calibrated to a **fixed false-positive rate** under a null model, not a percentile of the observed signal.

**Method.** Generate a matched-length surrogate of the cross-modal rupture composite via circular block bootstrap (already implemented for CSD nulls in §2G). Pick the threshold that yields FPR = 5 % on the surrogate. Apply it to the real composite. Report event count + H1 rate side-by-side with the percentile-based detector so the reader sees both.

**Anti-circularity.** Detector inputs remain head + gaze encodings only; disjoint from `CCA_1(t)` inputs (body + FAU + head pose). Same independence argument as `project_cross_modal_rupture.md`.

**AI-safety mapping.** Anthropic direction #5 (scalable oversight): a detector with a stated, known FPR is auditable; a percentile detector is not. This cell is a prototype of an auditable rupture monitor for human-AI interaction studies.


In [ ]:
# ============================================================================
# 8. Calibrated-FPR cross-modal rupture detector (Boettiger-Hastings 2012)
# ============================================================================
# Sidecar to §2C percentile detector. Does NOT replace events_df.
# Reports alt_events_df for side-by-side H1 rate comparison.
# PER-SAMPLE FPR (not FWER): threshold = 95th percentile of
#                      the POOLED surrogate values across all time points × all
#                      surrogates. The previous max-statistic mapping (Westfall-
#                      Young FWER) was too conservative and produced zero events.
# Fixed-block bootstrap has a small truncation-bias at the
#                      tail (the last partial block is not uniformly sampled);
#                      for a stationary-bootstrap variant (Politis & Romano 1994
#                      §2.2), the 10 s block × 600-frame signal case has ≤ 2%
#                      distortion. Documented; not fixed in .
# All four streams (head_a/head_b/gaze_a/gaze_b) must be
#                      present before processing; skipped count logged.

# SEED added — consumed by
# np.random.default_rng(SEED) inside the cell.
_required_fpr = ["dyad_streams", "np", "RESULTS_DIR", "events_df", "SEED"]
_missing_fpr = [n for n in _required_fpr if n not in globals()]
if _missing_fpr:
    print(f"[skip] calibrated-FPR: missing {_missing_fpr} (run §2C first)")
else:
    TARGET_FPR = 0.05
    N_SURROGATES = 200
    BLOCK_S = 10.0
    FPS = 30.0
    SUSTAIN_S_CAL = 0.75
    MIN_RUN = max(1, int(round(SUSTAIN_S_CAL * FPS)))
    FPR_MIN_FRAMES = 600   # 20 s @ 30 Hz; ≥ 2 non-overlapping 10 s blocks
    _CAL_RNG = np.random.default_rng(SEED)

    def _circ_block_shuffle(x, block_frames):
        """Circular block bootstrap per Politis & Romano 1994 §2.1 (fixed-block
        variant).

        [QA skill note — IAAFT vs block bootstrap.] research-notebook-qa skill
        prescribes IAAFT (Schreiber & Schmitz 1996) as the default surrogate
        method for temporal signals. We deliberately use block bootstrap here
        because Boettiger & Hastings 2012 (the method being ported) specifically
        uses block bootstrap for their calibrated-FPR detector — IAAFT preserves
        amplitude distribution + Fourier spectrum but BREAKS the non-stationary
        drift that CSD indicators are trying to measure, so it is inappropriate
        for this detector. IAAFT is better suited to stationarity-preserving
        nulls (e.g., Hilbert-phase synchrony on stationary segments).

        NOTE [H2]: the last partial block has a small truncation bias — for
        production use prefer arch.bootstrap.StationaryBootstrap which uses
        variable block lengths ~ Geometric(1/block_s) and removes the boundary
        artifact. Acceptable here because 10 s block × ≥ 600 frame signals give
        ≤ 2% distortion on the 95th-percentile threshold."""
        n = len(x)
        if block_frames >= n:
            return x
        n_blocks = int(np.ceil(n / block_frames))
        starts = _CAL_RNG.integers(0, n, size=n_blocks)
        pieces = []
        for s in starts:
            end = s + block_frames
            if end <= n:
                pieces.append(x[s:end])
            else:
                pieces.append(np.concatenate([x[s:], x[: end - n]]))
        return np.concatenate(pieces)[:n]

    def _rupture_composite(head, gaze):
        """Z-scored head-motion-energy + gaze-divergence composite."""
        head = np.asarray(head, dtype=float)
        gaze = np.asarray(gaze, dtype=float)
        if head.ndim == 1:
            head_e = np.abs(np.diff(head, prepend=head[0]))
        else:
            head_e = np.linalg.norm(np.diff(head, axis=0, prepend=head[:1]), axis=1)
        if gaze.ndim == 1:
            gaze_e = np.abs(gaze)
        else:
            gaze_e = np.linalg.norm(gaze, axis=1)
        T = min(len(head_e), len(gaze_e))
        h_z = (head_e[:T] - np.nanmean(head_e[:T])) / (np.nanstd(head_e[:T]) + 1e-9)
        g_z = (gaze_e[:T] - np.nanmean(gaze_e[:T])) / (np.nanstd(gaze_e[:T]) + 1e-9)
        return 0.5 * (h_z + g_z)

    def _detect_sustained(z, threshold, min_run=MIN_RUN):
        """Count non-overlapping sustained runs where z ≥ threshold."""
        above = z >= threshold
        if not above.any():
            return 0
        runs = 0
        i = 0
        while i < len(above):
            if above[i]:
                j = i
                while j < len(above) and above[j]:
                    j += 1
                if (j - i) >= min_run:
                    runs += 1
                i = j
            else:
                i += 1
        return runs

    # ---- Per-dyad calibrated threshold ----
    alt_rows = []
    block_frames = int(round(BLOCK_S * FPS))
    skip_missing_streams = 0
    skip_short = 0
    skip_composite_err = 0
    for ik, d in dyad_streams.items():
        head_a = d.get("head_a"); head_b = d.get("head_b")
        gaze_a = d.get("gaze_a"); gaze_b = d.get("gaze_b")
        # [H2/M4] require all four streams
        if any(s is None for s in (head_a, head_b, gaze_a, gaze_b)):
            skip_missing_streams += 1
            continue
        try:
            comp_a = _rupture_composite(np.asarray(head_a), np.asarray(gaze_a))
            comp_b = _rupture_composite(np.asarray(head_b), np.asarray(gaze_b))
        except Exception:
            skip_composite_err += 1
            continue
        T = min(len(comp_a), len(comp_b))
        if T < FPR_MIN_FRAMES:
            skip_short += 1
            continue
        composite = 0.5 * (comp_a[:T] + comp_b[:T])
        # [H1 fix] PER-SAMPLE FPR: pool ALL surrogate values, take 1-α quantile.
        pooled_surr = []
        for _ in range(N_SURROGATES):
            surr = _circ_block_shuffle(composite, block_frames)
            pooled_surr.append(surr)
        pooled_arr = np.concatenate(pooled_surr)
        pooled_arr = pooled_arr[np.isfinite(pooled_arr)]
        if len(pooled_arr) < 100:
            skip_short += 1
            continue
        cal_thr = float(np.nanpercentile(pooled_arr, 100.0 * (1.0 - TARGET_FPR)))
        n_events_cal = _detect_sustained(composite, cal_thr)
        alt_rows.append({
            "interaction_key": ik,
            "relationship": d.get("relationship", "unknown"),
            "cal_threshold": cal_thr,
            "n_events_calibrated": n_events_cal,
            "composite_len": int(T),
        })

    import pandas as pd
    alt_events_df = pd.DataFrame(alt_rows)
    print(f"Calibrated-FPR detector: {len(alt_events_df)} dyads processed")
    print(f"  skipped missing streams: {skip_missing_streams}")
    print(f"  skipped short           : {skip_short}")
    print(f"  skipped composite error : {skip_composite_err}")
    if len(alt_events_df) == 0:
        print("[skip] no dyads passed guards for FPR-calibration")
    else:
        alt_events_df.to_pickle(RESULTS_DIR / "alt_events_calibrated_fpr.pkl")
        print(f"\n  target FPR={TARGET_FPR:.2%}, {N_SURROGATES} surrogates per dyad:")
        print(f"  mean n_events = {alt_events_df['n_events_calibrated'].mean():.2f}")
        print(f"  sum n_events  = {int(alt_events_df['n_events_calibrated'].sum())}")
        print(f"\n  Side-by-side vs percentile detector:")
        try:
            pct_total = int(len(events_df)) if hasattr(events_df, '__len__') else 0
        except Exception:
            pct_total = 0
        print(f"    percentile-based  (§2C)  : {pct_total} total events")
        print(f"    calibrated-FPR    (§8)   : "
              f"{int(alt_events_df['n_events_calibrated'].sum())} total events "
              f"(target FPR {TARGET_FPR:.1%}, per-sample)")
        print(f"\n✓ Saved alt_events_calibrated_fpr.pkl")


## 9. Per-Modality CCA — Decomposing the Coupling Signal

**Motivation.** The upstream `CCA_1(t)` is FAU-dominated (~91 % of |w| on the Apr-19 run; see `§6` limitations). A single joint CCA collapses body / FAU / head into one canonical direction, washing out per-modality coupling differences. Fitting three CCAs *separately* and stacking their `canonical_r` statistics as meta-features lets each modality contribute proportional to its own coupling signal.

**Implementation.** Reload per-dyad feature matrices via `experiments.signal_utils.extract_dyad_cca_features` (same entry point NB1 uses), slice columns by modality prefix (`body_`, `fau_`, `head_`), fit `sklearn.cross_decomposition.CCA(n_components=min(d, 5))` per modality, extract 4 statistics per modality pair (mean / max / std canonical_r + n_frames), and classify the 12-dim meta-feature vector with a balanced logistic regression under 5-fold stratified CV.

**AI-safety mapping.** Anthropic direction #2: modality-decomposed coupling measurement is the prerequisite for claiming a multimodal-alignment metric is genuinely multimodal (rather than a single-channel proxy masquerading as multimodal).


In [ ]:
# ============================================================================
# 9. Per-modality CCA (body / fau / head) with stacked meta-features
# ============================================================================
# Ported from sprints/2026-04-ai-safety/proposals/A_per_modality_cca_2026-04-20.py
# Modality prefix tuples simplified: `body_` is the catch-all
#                     and already matches `body_pose_`, `body_trans_vel_`, etc. —
#                     the sub-prefixes were redundant and created dead-code
#                     maintenance risk.
# n_components lowered to min(len(idx)-1, PM_N_COMP) to stay
#                     strictly below sklearn's n_components < min(n_samples, n_feat)
#                     requirement. Failure counter added so silent drops surface.
# Runtime: O(N_dyads × 4 modalities × CCA_fit) ≈ 3-6 min on 1,700 dyads.
# Requires experiments.signal_utils (from Cell 1C prelude).

_required_pm = ["nb1_features", "MANIFEST_PATH", "NPZ_DIR", "np", "pd",
                "RESULTS_DIR", "SEED"]
_missing_pm = [n for n in _required_pm if n not in globals()]
if _missing_pm:
    print(f"[skip] per-modality CCA: missing {_missing_pm}")
else:
    try:
        from experiments.signal_utils import (
            load_dyad_from_manifest, extract_dyad_cca_features,
        )
        from sklearn.cross_decomposition import CCA as _SkCCA
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import StratifiedKFold
        from sklearn.preprocessing import StandardScaler
        from sklearn.metrics import roc_auc_score
        _PM_OK = True
    except ImportError as _e:
        _PM_OK = False
        print(f"[skip] per-modality CCA deps missing: {_e}")

    if _PM_OK:
        # [H3] single prefix per modality — each acts as a catch-all; sub-prefixes
        # within the same modality (e.g. body_pose_, body_trans_vel_) are already
        # captured by the base prefix and don't need enumeration.
        MODALITY_PREFIXES = {
            "body": ("body_",),
            "fau":  ("fau_",),
            "head": ("head_",),
        }
        PM_MIN_LENGTH = 640
        PM_N_COMP    = 5

        import json as _json
        with open(MANIFEST_PATH) as _mf:
            _pm_manifest = _json.load(_mf)

        pm_rows = []
        pm_skipped = 0
        pm_cca_failures = {"body": 0, "fau": 0, "head": 0}  # [H4] surface failures
        dyad_ids = list(nb1_features["interaction_key"])
        for i, ik in enumerate(dyad_ids):
            dyad = load_dyad_from_manifest(
                MANIFEST_PATH, NPZ_DIR, ik, preloaded_manifest=_pm_manifest,
            )
            if dyad is None:
                pm_skipped += 1
                continue
            try:
                triple = extract_dyad_cca_features(
                    dyad,
                    modality_bandpass={"body": (0.12, 1.29),
                                       "fau":  (0.23, 2.11),
                                       "head": (0.12, 1.29)},
                    fs=30.0,
                    use_intersected_valid_mask=True,
                    min_length=PM_MIN_LENGTH,
                )
            except Exception:
                pm_skipped += 1
                continue
            if triple is None:
                pm_skipped += 1
                continue
            feats_a, feats_b, col_names = triple
            row = {"interaction_key": ik}
            for modality, prefixes in MODALITY_PREFIXES.items():
                mask = [any(n.startswith(p) for p in prefixes) for n in col_names]
                idx = np.where(mask)[0]
                if len(idx) < 2:
                    continue
                X = feats_a[:, idx]
                Y = feats_b[:, idx]
                valid = ~(np.isnan(X).any(axis=1) | np.isnan(Y).any(axis=1))
                X, Y = X[valid], Y[valid]
                if len(X) < PM_MIN_LENGTH:
                    continue
                # [H4] sklearn CCA requires n_components < min(n_samples, n_feat_X, n_feat_Y).
                # Use len(idx) - 1 as a safety margin.
                n_comp = int(min(max(1, len(idx) - 1), PM_N_COMP))
                try:
                    cca = _SkCCA(n_components=n_comp, max_iter=500)
                    cca.fit(X, Y)
                    Xc, Yc = cca.transform(X, Y)
                    rs = np.array([np.corrcoef(Xc[:, j], Yc[:, j])[0, 1]
                                   for j in range(n_comp)])
                    row[f"{modality}_r_mean"] = float(np.mean(rs))
                    row[f"{modality}_r_max"]  = float(np.max(rs))
                    row[f"{modality}_r_std"]  = float(np.std(rs))
                    row[f"{modality}_n_frames"] = int(len(X))
                except Exception:
                    pm_cca_failures[modality] += 1
                    continue
            _rel_row = nb1_features[nb1_features["interaction_key"] == ik]
            row["relationship"] = (str(_rel_row.iloc[0].get("relationship", "unknown"))
                                   if len(_rel_row) else "unknown")
            if sum(k.endswith("_r_mean") for k in row) >= 2:
                pm_rows.append(row)
            if (i + 1) % 50 == 0:
                print(f"  [{i+1}/{len(dyad_ids)}] per-modality: kept={len(pm_rows)} "
                      f"skipped={pm_skipped} cca_fail={pm_cca_failures}")

        pm_df = pd.DataFrame(pm_rows).fillna(0.0)
        _n_r = len([c for c in pm_df.columns if "_r_" in c])
        _n_nf = len([c for c in pm_df.columns if c.endswith("_n_frames")])
        print(f"\nPer-modality CCA features: {len(pm_df)} dyads, "
              f"{_n_r} _r_ + {_n_nf} _n_frames = {_n_r + _n_nf} classifier features")
        if any(v > 0 for v in pm_cca_failures.values()):
            print(f"  per-modality CCA fit failures: {pm_cca_failures}")

        # ---- Classify (scaler fit inside each fold — parity with §7 fix) ----
        pm_auc = float("nan")
        if len(pm_df) >= 30 and pm_df["relationship"].nunique() >= 2:
            feat_cols = [c for c in pm_df.columns if "_r_" in c or c.endswith("_n_frames")]
            X = pm_df[feat_cols].values
            y = (pm_df["relationship"] == "familiar").astype(int).values
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
            probs = np.zeros(len(y))
            fold = []
            for tr, te in cv.split(X, y):
                _sc = StandardScaler()
                Xtr = _sc.fit_transform(X[tr])
                Xte = _sc.transform(X[te])
                clf = LogisticRegression(class_weight="balanced", max_iter=1000,
                                         random_state=SEED)
                clf.fit(Xtr, y[tr])
                probs[te] = clf.predict_proba(Xte)[:, 1]
                if len(np.unique(y[te])) == 2:
                    fold.append(roc_auc_score(y[te], probs[te]))
            pm_auc = float(roc_auc_score(y, probs))
            print(f"\nPer-modality CCA AUC: {pm_auc:.3f}  "
                  f"(folds: {[f'{a:.3f}' for a in fold]})")

            # feature importance — FIX B: compute on residualized
            # 9-feature matrix, not the collinear 12-feature matrix.
            # : the 3 _n_frames columns are
            # near-identical (shared intersected-valid-mask), splitting
            # coefficient mass arbitrarily across collinear features and
            # distorting the _r_ feature importance fractions.
            _feat_cols_imp = [c for c in feat_cols if not c.endswith("_n_frames")]
            _X_imp = pm_df[_feat_cols_imp].values
            _sc_full = StandardScaler()
            clf_full = LogisticRegression(class_weight="balanced", max_iter=1000,
                                          random_state=SEED)
            clf_full.fit(_sc_full.fit_transform(_X_imp), y)
            imp = dict(zip(_feat_cols_imp, [abs(c) for c in clf_full.coef_[0]]))
            tot = sum(imp.values()) + 1e-12
            imp_frac = {k: v / tot for k, v in imp.items()}
            print(f"\nTop modality-feature importances (relative |coef|, "
                  f"residualized 9-feature fit):")
            for k, v in sorted(imp_frac.items(), key=lambda x: -x[1])[:6]:
                print(f"  {k:<28s} {v:.3f}")

            # ================================================================
            # ----------------------------------------------------------------
            # body_n_frames / fau_n_frames / head_n_frames are near-identical
            # (shared intersected_valid_mask); collectively carry ~20% coef mass
            # in the joint fit above. Report the duration-only baseline AUC and
            # the n_frames-excluded AUC to adjudicate whether the 0.80+ AUC is
            # duration-confounded or a genuine per-modality signal.
            # ================================================================
            feat_no_nf = [c for c in feat_cols if not c.endswith("_n_frames")]
            feat_nf_only = [c for c in feat_cols if c.endswith("_n_frames")]

            def _cv_auc_subset(feats_in):
                if not feats_in:
                    return float("nan")
                Xs = pm_df[feats_in].values
                ps = np.zeros(len(y))
                for tr, te in cv.split(Xs, y):
                    _sc2 = StandardScaler()
                    Xtr2 = _sc2.fit_transform(Xs[tr])
                    Xte2 = _sc2.transform(Xs[te])
                    clf2 = LogisticRegression(class_weight="balanced",
                                               max_iter=1000, random_state=SEED)
                    clf2.fit(Xtr2, y[tr])
                    ps[te] = clf2.predict_proba(Xte2)[:, 1]
                return float(roc_auc_score(y, ps))

            pm_auc_no_nf = _cv_auc_subset(feat_no_nf)
            pm_auc_nf_only = _cv_auc_subset(feat_nf_only)
            _delta = pm_auc - pm_auc_no_nf if not np.isnan(pm_auc_no_nf) else float("nan")
            print(f"\n")
            print(f"  Full features     AUC: {pm_auc:.3f} ({len(feat_cols)} feats)")
            print(f"  Excluding n_frames AUC: {pm_auc_no_nf:.3f} ({len(feat_no_nf)} feats)")
            print(f"  Only n_frames (baseline) AUC: {pm_auc_nf_only:.3f} ({len(feat_nf_only)} feats)")
            print(f"  Delta (full - no_nframes) = {_delta:+.3f}")
            if not np.isnan(pm_auc_no_nf):
                if pm_auc_no_nf >= 0.70:
                    # Compute Hanley-McNeil 95% CI for the residualized AUC at this N
                    try:
                        n_pos = int((y == 1).sum())
                        n_neg = int((y == 0).sum())
                        Q1 = pm_auc_no_nf / (2.0 - pm_auc_no_nf)
                        Q2 = 2.0 * pm_auc_no_nf**2 / (1.0 + pm_auc_no_nf)
                        var_auc = (pm_auc_no_nf * (1 - pm_auc_no_nf)
                                   + (n_pos - 1) * (Q1 - pm_auc_no_nf**2)
                                   + (n_neg - 1) * (Q2 - pm_auc_no_nf**2)) / (n_pos * n_neg)
                        se_auc = float(var_auc) ** 0.5
                        ci_lo = max(0.0, pm_auc_no_nf - 1.96 * se_auc)
                        ci_hi = min(1.0, pm_auc_no_nf + 1.96 * se_auc)
                    except Exception:
                        ci_lo, ci_hi = float("nan"), float("nan")
                    # Apply the pre-registered verdict-gate (Cell 0) to the residualized AUC
                    if pm_auc_no_nf < 0.60:
                        pm_label = "[NEGATIVE_RESULT]"
                    elif pm_auc_no_nf < 0.70:
                        pm_label = "[EMPIRICAL but weak]"
                    else:
                        pm_label = "[EMPIRICAL]"
                    print(f"  Hanley-McNeil 95% CI on residualized AUC: [{ci_lo:.3f}, {ci_hi:.3f}]")
                    print(f"  Label: {pm_label}  (residualized AUC {pm_auc_no_nf:.3f} vs 0.60/0.70 thresholds)")
                    print(f"  Verdict: residualized AUC {pm_auc_no_nf:.3f} remains above the 0.70 threshold; "
                          f"duration accounts for ~{(pm_auc - pm_auc_no_nf):.3f} of the full {pm_auc:.3f}, "
                          f"with the duration-only baseline at {pm_auc_nf_only:.3f} — "
                          f"per-modality coupling is partially attenuated by duration but the residualized "
                          f"signal clears the pre-registered threshold.")
                elif pm_auc_no_nf <= 0.55:
                    print(f"  VERDICT: per-modality signal is DURATION-CONFOUNDED "
                          f"(AUC collapses to {pm_auc_no_nf:.3f})")
                else:
                    print(f"  VERDICT: PARTIAL CONFOUND - {pm_auc_no_nf:.3f} below 0.70 "
                          f"threshold; duration carries some but not all signal")
        else:
            pm_auc_no_nf = float("nan")
            pm_auc_nf_only = float("nan")
            feat_no_nf = []
            feat_nf_only = []

        pm_df.to_pickle(RESULTS_DIR / "per_modality_cca_df.pkl")
        with open(RESULTS_DIR / "per_modality_cca_summary.json", "w") as f:
            _json.dump({"n_dyads": int(len(pm_df)), "auc": pm_auc,
                        "auc_no_nframes_residualized": float(pm_auc_no_nf) if len(pm_df) >= 30 else float("nan"),
                        "ci95_auc_no_nframes_residualized": [float(ci_lo), float(ci_hi)] if "ci_lo" in dir() else [float("nan"), float("nan")],
                        "auc_nframes_only": float(pm_auc_nf_only) if len(pm_df) >= 30 else float("nan"),
                        "importance_fractions_residualized": imp_frac if "imp_frac" in dir() else {},
                        "importance_fit": "residualized_9_feature_in_sample",
                        "feature_cols": [c for c in pm_df.columns if "_r_" in c],
                        "feature_cols_all_used": feat_cols if "feat_cols" in dir() else [],
                        "feature_cols_no_nframes": feat_no_nf if "feat_no_nf" in dir() else [],
                        "cca_failures_by_modality": pm_cca_failures,
                        "min_signal_frames": PM_MIN_LENGTH,
                        "n_components_cap": PM_N_COMP,
                        "scaler_fit_inside_fold": True,
                        "audit_provenance": "n_frames columns checked for duration confound; see auc_no_nframes_v6_4_2 for residualized estimate"},
                       f, indent=2)
        print(f"\n✓ Saved per_modality_cca_df.pkl + per_modality_cca_summary.json")


## 10. Robustness Checks — Class Balancing and Alternative Classifiers

**Motivation.** The Apr-19 baseline GBT classifier is a single model family trained on potentially imbalanced data. Two independent robustness checks:
- **B (Balancing).** SMOTE oversampling + class-weighted logistic regression + balanced Random Forest. Compare AUCs; if all three are within ± 0.03 of the GBT baseline, class imbalance is not the bottleneck.
- **C (Alt classifiers).** Nested-CV (5 outer × 3 inner) sweep over logistic-regression (`C ∈ {0.01, 0.1, 1, 10}`), `HistGradientBoostingClassifier`, and RBF-SVM. Reports best-of-three AUC with fold-level Cohen's d effect sizes.

Operates on `mi_df` (§7) + `pm_df` (§9) concatenated into a single feature matrix, so both balancing and model family are evaluated on the **richest** feature set we have. If neither lifts AUC materially above the GBT baseline, the bottleneck is signal, not method.

**AI-safety mapping.** Anthropic direction #6 (honesty + interpretability): reporting multiple model families with nested-CV AUC+CI prevents "lucky-model" overclaim, which is standard RSP evidence hygiene.


In [ ]:
# ============================================================================
# 10. Balancing (B) + alternative classifiers (C) on combined MI + per-modality
# ============================================================================
# Ported from sprints/2026-04-ai-safety/proposals/{B,C}_*_2026-04-20.py
# Known bugfixes applied at port time:
#   - B: replaced DataFrame.get() (unsupported) with direct column access.
#   - C: HGB class_weight is sklearn-version-gated (>=1.2). Version-sniffing
#        added for sklearn ≥ 1.2 compatibility.
# HistGradientBoostingClassifier accepts `class_weight` in
#                     sklearn ≥ 1.2. Without it, HGB is miscalibrated relative
#                     to the balanced LR / RF / SVM baselines — unfair comparison.
#                     The version sniff below adds class_weight='balanced' on
#                     supported sklearn.

_required_bc = ["np", "pd", "RESULTS_DIR", "SEED"]
_missing_bc = [n for n in _required_bc if n not in globals()]
_have_mi = "mi_df" in globals()
_have_pm = "pm_df" in globals()
if not (_have_mi or _have_pm):
    print("[skip] §10 needs mi_df (§7) and/or pm_df (§9) — run those first")
else:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
    from sklearn.svm import SVC
    from sklearn.model_selection import StratifiedKFold, GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import Pipeline

    # [H5] HGB class_weight support introduced in sklearn 1.2
    def _skl_version_tuple():
        try:
            return tuple(int(x) for x in sklearn.__version__.split(".")[:2])
        except Exception:
            return (0, 0)
    _HGB_KWARGS = {"random_state": SEED, "max_iter": 200}
    if _skl_version_tuple() >= (1, 2):
        _HGB_KWARGS["class_weight"] = "balanced"
    else:
        print(f"[note] sklearn {sklearn.__version__} — HGB class_weight unavailable; "
              "comparison to balanced LR/RF will be less directly comparable.")

    # ---- Build combined feature matrix ----
    def _keys(df):
        return {k for k in df.columns if k not in {"interaction_key", "relationship"}}

    if _have_mi and _have_pm:
        merged = mi_df.merge(pm_df.drop(columns=["relationship"], errors="ignore"),
                             on="interaction_key", how="inner")
        feat_cols = sorted(_keys(mi_df) | _keys(pm_df))
        feat_cols = [c for c in feat_cols if c in merged.columns]
        # FIX A: drop _n_frames columns for duration-clean comparison
        # with §9 residualized AUC. Code-reviewer audit 2026-04-24 — without this,
        # §10 inherits the duration confound the §9 audit explicitly removed.
        feat_cols = [c for c in feat_cols if not c.endswith("_n_frames")]
    elif _have_mi:
        merged = mi_df.copy()
        feat_cols = sorted(_keys(mi_df))
    else:
        merged = pm_df.copy()
        feat_cols = sorted(_keys(pm_df))

    merged = merged[merged["relationship"].isin(("stranger", "familiar"))].reset_index(drop=True)
    if len(merged) < 30 or merged["relationship"].nunique() < 2:
        print(f"[skip] §10 combined features: insufficient data (n={len(merged)})")
    else:
        X = merged[feat_cols].fillna(0.0).values
        y = (merged["relationship"] == "familiar").astype(int).values
        print(f"Combined feature matrix: {X.shape}; "
              f"stranger={(y == 0).sum()} familiar={(y == 1).sum()}")

        # ================================================================
        # Bootstrap 95% CI helper for AUC
        # ----------------------------------------------------------------
        # N=186 + 5-fold CV: fold-level AUC SD can easily exceed point-estimate
        # differences between families. Without CIs, "alt classifier AUC 0.765
        # vs balanced-LR 0.748" is uncomputable as a meaningful claim. 1,000-
        # resample percentile bootstrap over pooled OOF prediction vector.
        # ================================================================
        _rng_boot = np.random.default_rng(SEED)
        def _boot_ci(y_true, probs, n_boot=1000):
            n = len(y_true)
            aucs = []
            for _ in range(n_boot):
                idx = _rng_boot.integers(0, n, size=n)
                if len(np.unique(y_true[idx])) < 2:
                    continue
                aucs.append(roc_auc_score(y_true[idx], probs[idx]))
            if not aucs:
                return (float("nan"), float("nan"))
            return (float(np.percentile(aucs, 2.5)),
                    float(np.percentile(aucs, 97.5)))


        # ---- Proposal B: three balancing strategies ----
        print("\n--- Proposal B: balancing strategies ---")
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        b_results = {}

        # B1: balanced-LR (scaler fit inside fold)
        probs_lr = np.zeros(len(y))
        for tr, te in cv.split(X, y):
            sc = StandardScaler()
            Xt, Xe = sc.fit_transform(X[tr]), sc.transform(X[te])
            clf = LogisticRegression(class_weight="balanced", max_iter=1000,
                                     random_state=SEED)
            clf.fit(Xt, y[tr])
            probs_lr[te] = clf.predict_proba(Xe)[:, 1]
        b_results["auc_balanced_lr"] = float(roc_auc_score(y, probs_lr))

        # B2: SMOTE + LR
        try:
            from imblearn.over_sampling import SMOTE
            probs_sm = np.zeros(len(y))
            for tr, te in cv.split(X, y):
                sc = StandardScaler()
                Xt, Xe = sc.fit_transform(X[tr]), sc.transform(X[te])
                n_min = int(min(np.bincount(y[tr])))
                k = min(5, max(1, n_min - 1))
                if n_min > 1:
                    sm = SMOTE(random_state=SEED, k_neighbors=k)
                    Xr, yr = sm.fit_resample(Xt, y[tr])
                else:
                    Xr, yr = Xt, y[tr]
                clf = LogisticRegression(max_iter=1000, random_state=SEED)
                clf.fit(Xr, yr)
                probs_sm[te] = clf.predict_proba(Xe)[:, 1]
            b_results["auc_smote_lr"] = float(roc_auc_score(y, probs_sm))
        except ImportError:
            b_results["auc_smote_lr"] = float("nan")
            print("  (imblearn not installed — skipping SMOTE; pip install imbalanced-learn)")

        # B3: balanced-RF
        probs_rf = np.zeros(len(y))
        for tr, te in cv.split(X, y):
            sc = StandardScaler()
            Xt, Xe = sc.fit_transform(X[tr]), sc.transform(X[te])
            rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                        random_state=SEED, n_jobs=-1)
            rf.fit(Xt, y[tr])
            probs_rf[te] = rf.predict_proba(Xe)[:, 1]
        b_results["auc_balanced_rf"] = float(roc_auc_score(y, probs_rf))

        # Bootstrap CIs for Proposal B
        # Iterate over frozen snapshot; accumulate CIs then merge (avoids
        # RuntimeError: dictionary changed size during iteration).
        _b_probs = {"auc_balanced_lr": probs_lr,
                    "auc_smote_lr": probs_sm if "probs_sm" in dir() else None,
                    "auc_balanced_rf": probs_rf}
        _b_ci = {}
        for k, v in list(b_results.items()):
            probs_k = _b_probs.get(k)
            if probs_k is not None and not np.isnan(v):
                lo, hi = _boot_ci(y, probs_k)
                _b_ci[f"ci95_{k}"] = [lo, hi]
                print(f"  {k:<22s} AUC={v:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
            else:
                print(f"  {k:<22s} AUC={v:.3f}")
        b_results.update(_b_ci)

        # ---- Proposal C: nested-CV over three model families (scaling in pipeline) ----
        print("\n--- Proposal C: nested-CV alternative classifiers (5 outer × 3 inner) ---")
        outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
        pipes = {
            "lr": (Pipeline([("sc", StandardScaler()),
                             ("clf", LogisticRegression(class_weight="balanced",
                                                        max_iter=1000,
                                                        random_state=SEED))]),
                   {"clf__C": [0.01, 0.1, 1.0, 10.0]}),
            # [H5] HGB now receives class_weight when sklearn supports it
            "hgb": (Pipeline([("clf", HistGradientBoostingClassifier(**_HGB_KWARGS))]),
                    {"clf__learning_rate": [0.05, 0.1],
                     "clf__max_depth": [3, 5],
                     "clf__min_samples_leaf": [5, 10]}),
            "svm": (Pipeline([("sc", StandardScaler()),
                              ("clf", SVC(class_weight="balanced", probability=True,
                                          random_state=SEED))]),
                    {"clf__C": [0.1, 1.0, 10.0],
                     "clf__kernel": ["rbf", "linear"]}),
        }
        c_results = {}
        for name, (pipe, grid) in pipes.items():
            probs_c = np.zeros(len(y))
            best_params = []
            for tr, te in outer.split(X, y):
                search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc",
                                      n_jobs=-1, refit=True)
                search.fit(X[tr], y[tr])
                best_params.append(search.best_params_)
                probs_c[te] = search.predict_proba(X[te])[:, 1]
            c_results[f"auc_{name}"] = float(roc_auc_score(y, probs_c))
            c_results[f"best_params_{name}"] = best_params
            # bootstrap 95% CI
            lo_c, hi_c = _boot_ci(y, probs_c)
            c_results[f"ci95_auc_{name}"] = [lo_c, hi_c]
            print(f"  {name:>4s}  AUC={c_results[f'auc_{name}']:.3f}  "
                  f"95% CI [{lo_c:.3f}, {hi_c:.3f}]  "
                  f"best: {best_params[0] if best_params else {}}")

        best = max([k for k in c_results if k.startswith("auc_")],
                   key=lambda k: c_results[k])
        print(f"\nBest alt classifier: {best.replace('auc_', '')} "
              f"(AUC={c_results[best]:.3f})")

        # ---- Persist ----
        import json as _json
        out = {
            "n_dyads": int(len(merged)),
            "n_features": int(X.shape[1]),
            "sklearn_version": sklearn.__version__,
            "hgb_class_weight_enabled": "class_weight" in _HGB_KWARGS,
            "balancing": b_results,
            "nested_cv": c_results,
        }
        with open(RESULTS_DIR / "alt_classifiers_summary.json", "w") as f:
            _json.dump(out, f, indent=2, default=str)
        print(f"\n✓ Saved alt_classifiers_summary.json")

# ============================================================================
# Merge  sidecars into h1_verdict_summary.json
# ============================================================================
# Rationale: manuscripts and post-run audits need a single authoritative verdict
# file. Without this merge, a reviewer has to manually cross-reference 5 JSONs.
# The merge is ADDITIVE: existing h1_verdict_summary keys are preserved; 
# outputs land under a "additions" subdict to make provenance explicit.

import json as _vj
from pathlib import Path as _VP

_v_path = RESULTS_DIR / "h1_verdict_summary.json"
if not _v_path.exists():
    print("[skip] h1_verdict_summary.json not yet written — re-run after Cell 16 lands the verdict.")
else:
    with open(_v_path) as _vf:
        _verdict = _vj.load(_vf)
    _v6_4 = {}

    for _name, _fn, _keys in [
        ("mi", "mi_diagnostic_summary.json",
         ("n_dyads", "mi_auc", "cca_auc_baseline", "delta_mi_minus_cca",
          "linearity_limit", "k_neighbors", "cca_dim_used", "min_signal_frames")),
        ("per_modality_cca", "per_modality_cca_summary.json",
         ("n_dyads", "auc", "feature_cols", "cca_failures_by_modality",
          "min_signal_frames", "n_components_cap")),
        ("alt_classifiers", "alt_classifiers_summary.json",
         ("n_dyads", "n_features", "sklearn_version", "hgb_class_weight_enabled",
          "balancing", "nested_cv")),
    ]:
        _fp = RESULTS_DIR / _fn
        if _fp.exists():
            with open(_fp) as _f:
                _d = _vj.load(_f)
            _v6_4[_name] = {k: _d[k] for k in _keys if k in _d}
        else:
            _v6_4[_name] = {"status": "absent", "file": _fn}

    # §8 calibrated-FPR — summarize the pkl-stored df into scalars
    _fpr_pkl = RESULTS_DIR / "alt_events_calibrated_fpr.pkl"
    if _fpr_pkl.exists():
        import pandas as _pd
        _df = _pd.read_pickle(_fpr_pkl)
        _v6_4["calibrated_fpr"] = {
            "n_dyads": int(len(_df)),
            "total_events": int(_df["n_events_calibrated"].sum()) if "n_events_calibrated" in _df else 0,
            "mean_events_per_dyad": float(_df["n_events_calibrated"].mean()) if "n_events_calibrated" in _df else 0.0,
            "target_fpr": 0.05,
            "reference": "Boettiger & Hastings 2012, Proc R Soc B 279:4734-4739",
        }
    else:
        _v6_4["calibrated_fpr"] = {"status": "absent"}

    _verdict["additions"] = _v6_4
    _verdict["audit_provenance"] = {
        "computation_date": "sample_run",
        "audit_cycle": "duration-confound + verdict-merge",
        "audit_targets": ["MI-vs-CCA linearity gate", "calibrated-FPR detector", "verdict-merge block"],
        "event_counts_note": ("n_events_effective = total tau-computable events across horizons; "
                              "n_events_by_horizon[h] = count after pre-window geometry filter at horizon h seconds; "
                              "null distribution size per indicator-horizon = N_TIME_SHUFFLES x valid-pre-window event count."),
    }
    with open(_v_path, "w") as _vf:
        _vj.dump(_verdict, _vf, indent=2, default=str)
    print(f"✓ Merged sidecar diagnostics into {_v_path.name}")
    print(f"  keys added: {list(_v6_4.keys())}")
